# Faster R-CNN (ResNet-50-FPN V2) + AI Hub 외부 데이터(외부 전용 약 새 클래스) + 각인 OCR

`faster_rcnn_jang2.ipynb`를 복사해 두 가지를 바꾼 실험이다. 탐지기 하이퍼파라미터·라벨·분할·평가는 그대로다.

| 항목 | faster_rcnn_jang2 | 이 노트북 |
|---|---|---|
| 외부 데이터 가져오기 | 내 Drive ZIP을 Colab 로컬로 복사, 02_eda 검수 CSV 또는 자동 전처리 | `exp/yolo11m_100_aihub_not_background`와 같다. Kaggle·Colab은 `AIHUB_URL` Drive 폴더에서 zip을 받아 풀거나 이미 푼 폴더(Kaggle 입력 데이터셋)를 쓰고, 로컬은 미리 푼 폴더를 쓴다. 선별·검사는 02_eda §5~8 |
| 외부 전용 약 (대회 56종에 없는 약) | 라벨 없이 배경 | 56 뒤 번호의 새 클래스로 학습(`extra_as_class`). 평가에서는 빼고, 제출에는 약 코드를 category_id로 넣는다(`submit_extra`, B안) |
| 각인 OCR | 없음 | 탐지기 학습 뒤 같은 학습 사진의 정답 bbox crop으로 CRNN+CTC를 따로 학습하고, 탐지 bbox의 클래스를 OCR로 다시 고른다(`exp/yolo26m_nms_free_aihub_copypaste_eda_ocr` 5-2절과 같은 방법) |

- 대회 데이터의 라벨·분할·학습·평가는 이 노트북 안에서 처리한다. 라벨 txt·분할 목록·`data.yaml` 파일은 만들지 않는다.
- 라벨: 원본 JSON + 통합 보정 기록(`data/processed/corrections.csv`)의 v1 기록을 노트북 안에서 적용한다.
- 평가: AP 계산은 ultralytics 검증과 같은 방식(클래스가 같은 정답과만 짝, 101점 보간)으로 직접 구현한다.
  val에는 대회 사진만 있으므로 외부 전용 클래스 예측은 평가에서 뺀다.
- 결과: `runs/<exp>/results/`에 YOLO `evaluate_fold`와 같은 열의 `*_summary.csv`, `*_per_class.csv`를
  저장하므로 YOLO 결과와 한 표로 모을 수 있다. OCR 결과는 `runs/<exp>/results/ocr/<run>/`에 남긴다.

**위험**: 채점 정답에 외부 전용 약이 없거나 번호 체계가 다르면 그 약으로 낸 박스는 틀린 박스가 된다.
val로는 확인할 수 없고 리더보드로만 확인된다. `submit_extra = False`면 대회 56종만 제출한다(A안).

순서: 0 Kaggle·Colab 준비 → 1 설정 → 2 라벨 → 3 분할 → 4 데이터셋(4-1 외부 데이터) → 5 모델 → 6 평가 지표 → 7 예측·평가
→ 8 학습 루프 → 9 동작 확인 → 10 본 학습 → 11 요약 → 12 학습 곡선 → 13 혼동행렬 → 14 실패 사례
→ 15 각인 OCR → 16 제출

## 0. Kaggle·Colab 준비 (Kaggle·Colab에서만 실행)

Kaggle은 노트북 설정에서 Accelerator를 **GPU T4**, Internet을 **On**으로 켜고, Colab은 런타임 유형을 GPU로 고른 뒤 실행한다.
이 노트북 파일만 올리면 된다.

- 공개 저장소 `def-uk/pill-bbox`의 `REPO_BRANCH`를 Kaggle은 `/kaggle/working/pill-bbox`, Colab은 `/content/pill-bbox`에 받는다. 이미 받았으면 새 커밋만 받는다.
  이후 셀은 받은 저장소의 `data_process`와 `data/processed/corrections.csv`를 쓴다.
- 기본 이미지에 없을 수 있는 `gdown`, `python-dotenv`만 설치한다. torch·torchvision은 Kaggle·Colab에 설치된 것을 쓴다.
- 대회 원본은 Kaggle 입력 데이터셋으로 붙인다. 1절이 `/kaggle/input`에서 `train_images`·`test_images`·`train_annotations`가
  함께 든 폴더를 찾는다.
- 로컬에서는 아무것도 하지 않는다.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/def-uk/pill-bbox.git"  # 공개 저장소라 토큰이 필요 없다
REPO_BRANCH = "exp/faster_rcnn_aihub_ocr"  # 이 노트북이 올라가 있는 브랜치
IN_KAGGLE = Path("/kaggle").is_dir()
IN_COLAB = not IN_KAGGLE and (bool(os.getenv("COLAB_RELEASE_TAG")) or Path("/content").is_dir())
REPO_DIR = Path("/kaggle/working/pill-bbox") if IN_KAGGLE else Path("/content/pill-bbox")

if IN_KAGGLE or IN_COLAB:
    if (REPO_DIR / ".git").is_dir():
        # 이미 받았으면 새 커밋만 받는다. 고친 파일과 겹치면 덮어쓰지 않고 멈춘다.
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    elif REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR}가 있지만 git 저장소가 아닙니다. 폴더를 지우고 다시 실행하세요."
        )
    else:
        print(f"저장소를 받습니다: {REPO_URL} ({REPO_BRANCH}) → {REPO_DIR}")
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                REPO_BRANCH,
                REPO_URL,
                str(REPO_DIR),
            ],
            check=True,
        )
    commit = subprocess.run(
        ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h %s"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    print(f"저장소 커밋: {commit}")
    # 기본 이미지에 없을 수 있는 패키지만 설치한다.
    packages = {"gdown": "gdown", "python-dotenv": "dotenv"}
    missing = [
        name for name, module in packages.items() if importlib.util.find_spec(module) is None
    ]
    if missing:
        print(f"패키지 설치: {missing}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    os.chdir(REPO_DIR)
else:
    print("Kaggle·Colab이 아니므로 건너뜁니다.")

## 1. 설정

하이퍼파라미터는 `FASTER_RCNN_REQUIREMENTS.md` 4절에서 확정한 값이다. 코드에 고정하지 않고 `CFG`로
함수에 넘기며, 실행한 설정 전체를 run 폴더의 `config.json`에 남긴다.
외부 데이터 설정은 4-1절의 `CFG["external_data"]`, 각인 OCR 설정은 `CFG["ocr"]`이다.

**Kaggle에서 실행**: 0절이 저장소를 `/kaggle/working/pill-bbox`에 받는다. 대회 원본은 입력 데이터셋에서
자동으로 찾고(`find_kaggle_raw_dir`), 후보가 여러 개면 `RAW_DIR_OVERRIDE`에 경로를 적는다.
`runs_dir` 기본값(`/kaggle/working/pill-bbox/runs`)은 커밋 출력으로 남는다(`/kaggle/working` 20GB 한도).
`AIHUB_URL`은 Add-ons → Secrets에 같은 이름으로 등록한다(Save & Run All은 입력창을 띄울 수 없다).
`runs_dir`는 Colab이면 내 Drive(`/content/drive/MyDrive/pill-bbox/runs`)를 연결해 쓰고, Kaggle·로컬이면 저장소의 `runs`다.
같은 run 폴더를 옮기면 Kaggle에서 학습하고 Colab에서 이어서 평가·OCR·제출할 수 있다(반대도 같다).

In [ ]:
import os

# MPS에 없는 연산이 있으면 CPU로 넘긴다. torch import 전에 정해야 한다.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import contextlib
import json
import math
import platform
import random
import sys
import time
import warnings
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
from matplotlib import font_manager
from matplotlib.patches import Rectangle
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet50_Weights
from torchvision.models.detection import (
    FasterRCNN_ResNet50_FPN_V2_Weights,
    fasterrcnn_resnet50_fpn_v2,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm


def find_project_root(start: Path) -> Path:
    """start에서 위로 올라가며 pyproject.toml이 있는 폴더를 찾는다."""
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").is_file():
            return path
    raise FileNotFoundError(f"{start} 위에서 pyproject.toml을 찾지 못했습니다.")


# 실행 환경: Kaggle(GPU)·Colab·로컬. Kaggle은 저장소를 /kaggle/working/pill-bbox에 clone한다.
IN_KAGGLE = Path("/kaggle").is_dir()
IN_COLAB = not IN_KAGGLE and (bool(os.getenv("COLAB_RELEASE_TAG")) or Path("/content").is_dir())


def locate_project_root() -> Path:
    """현재 폴더 → Kaggle → Colab 저장소 위치 순서로 pyproject.toml이 있는 폴더를 찾는다."""
    for start in [Path.cwd(), Path("/kaggle/working/pill-bbox"), Path("/content/pill-bbox")]:
        with contextlib.suppress(FileNotFoundError):
            return find_project_root(start)
    raise FileNotFoundError(
        "저장소를 찾지 못했습니다. Kaggle은 /kaggle/working/pill-bbox에 clone하세요."
    )


PROJECT_ROOT = locate_project_root()
RAW_SUBDIRS = ("train_images", "test_images", "train_annotations")


def find_kaggle_raw_dir(input_root: Path = Path("/kaggle/input")) -> Path:
    """Kaggle 입력 데이터셋에서 대회 원본 세 폴더가 함께 든 폴더를 찾는다(4단계 깊이까지)."""
    found = sorted(
        {
            path.parent
            for depth in range(4)
            for path in input_root.glob("*/" * depth + "train_images")
            if all((path.parent / name).is_dir() for name in RAW_SUBDIRS)
        }
    )
    if len(found) != 1:
        raise FileNotFoundError(
            f"{input_root}에서 {RAW_SUBDIRS}가 함께 든 폴더를 하나로 정할 수 없습니다: {found}. "
            "대회 데이터를 입력 데이터셋으로 붙이고, 여러 개면 RAW_DIR_OVERRIDE에 적으세요."
        )
    return found[0]


# 대회 원본(train_images, test_images, train_annotations) 폴더. None이면 저장소의 data/raw를 쓰고,
# Kaggle에서 그 폴더가 없으면 입력 데이터셋에서 찾는다. 직접 정하려면 경로를 적는다.
RAW_DIR_OVERRIDE: str | None = None
if RAW_DIR_OVERRIDE:
    RAW_DIR = Path(RAW_DIR_OVERRIDE)
elif IN_KAGGLE and not (PROJECT_ROOT / "data" / "raw" / "train_images").is_dir():
    RAW_DIR = find_kaggle_raw_dir()
else:
    RAW_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_IMG_DIR = RAW_DIR / "train_images"
TEST_IMG_DIR = RAW_DIR / "test_images"
ANN_DIR = RAW_DIR / "train_annotations"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


def default_runs_dir() -> Path:
    """학습 결과 폴더. Colab은 런타임이 끝나도 남도록 내 Drive, Kaggle·로컬은 저장소의 runs."""
    if not IN_COLAB:
        return PROJECT_ROOT / "runs"  # Kaggle은 /kaggle/working 아래라 커밋 출력으로 남는다
    from google.colab import drive

    drive.mount("/content/drive")
    return Path("/content/drive/MyDrive/pill-bbox/runs")


CFG = {
    "exp": "frcnn_aihub_ocr",
    "runs_dir": str(default_runs_dir()),
    # 데이터
    "label_version": "v1",
    "split_unit": "combo",
    "fix_rare": False,  # True면 희귀 약이 든 조합을 모든 fold에서 train에 고정(combofix 분할)
    "n_folds": 5,
    "folds": [0, 1, 2, 3, 4],
    "fraction": 1.0,  # train 사진 중 이 비율만 사용(동작 확인용)
    "seed": 42,
    # 모델
    "pretrained": "coco",  # coco: COCO_V1 전체 / imagenet: 백본만 ImageNet
    "trainable_backbone_layers": 5,  # 5 = 전체 파인튜닝
    "min_size": 800,
    "max_size": 1333,
    # 학습
    "epochs": 100,
    "batch": 4,
    "optimizer": "SGD",
    "lr0": 0.001,
    "momentum": 0.9,
    "weight_decay": 1e-4,
    "warmup_epochs": 3.0,  # 이 기간 동안 lr을 lr0 × warmup_factor에서 lr0까지 선형으로 올림
    "warmup_factor": 0.001,
    "cos_lr": False,  # True: 코사인 감소 / False: 선형 감소(워밍업 이후)
    "lrf": 0.01,  # 마지막 학습률 = lr0 × lrf = 0.00001
    "hflip_p": 0.0,  # 증강은 나중에. 0이면 증강 없음
    "amp": True,  # cuda에서만 켜진다
    "patience": 10,  # val mAP75-95가 이 epoch 수만큼 좋아지지 않으면 멈춤
    "workers": 0 if sys.platform == "darwin" else 2,  # mac 노트북은 spawn 문제로 0
    # 평가: YOLO evaluate_fold와 같은 조건
    "conf": 0.001,  # torchvision 기본 box_score_thresh=0.05 대신
    "max_det": 300,  # torchvision 기본 box_detections_per_img=100 대신
    "nms_iou": 0.5,  # torchvision 기본 box_nms_thresh (클래스별 NMS)
    "eval_batch": 4,
    # 각인 OCR (15절). 탐지기 학습에는 영향이 없고, 학습이 끝난 뒤 따로 학습해 클래스를 다시 고른다.
    "ocr": {
        "enabled": True,
        "imgsz": 128,  # OCR 입력 한 변(px). 정답 bbox의 긴 변을 이 크기로 맞춘다
        "epochs": 30,
        "batch": 128,
        "lr": 0.001,  # AdamW 최대 학습률(one-cycle 코사인)
        "patience": 10,  # 검증 각인 분류 정확도가 이 epoch 동안 오르지 않으면 멈춤
        "rotations": 8,  # 추론 때 crop을 360/이 값 도씩 돌려 가장 잘 읽히는 방향을 쓴다
        "temperature": 1.0,  # 클래스별 CTC 로그 확률을 softmax하기 전에 나누는 값
        "switch_prob": 0.9,  # OCR 최고 클래스 확률이 이 값 이상이고
        "keep_prob": 0.05,  # 탐지기가 고른 클래스의 OCR 확률이 이 값 이하일 때만 클래스를 바꾼다
    },
}

In [ ]:
def get_device() -> str:
    """CUDA > MPS > CPU 순서로 장치를 고른다(src.config.get_device와 같은 규칙)."""
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def sync(device: str) -> None:
    """시간을 잴 때 비동기로 도는 GPU 연산이 끝날 때까지 기다린다."""
    if device == "cuda":
        torch.cuda.synchronize()
    elif device == "mps":
        torch.mps.synchronize()


def seed_everything(seed: int) -> None:
    """python·numpy·torch 난수를 고정한다."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def set_korean_font() -> None:
    """설치된 한글 글꼴이 있으면 matplotlib 기본 글꼴로 쓴다."""
    installed = {f.name for f in font_manager.fontManager.ttflist}
    for name in ["AppleGothic", "NanumGothic", "Malgun Gothic", "Noto Sans CJK KR"]:
        if name in installed:
            plt.rcParams["font.family"] = name
            break
    plt.rcParams["axes.unicode_minus"] = False


DEVICE = get_device()
set_korean_font()
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"device: {DEVICE}, torch {torch.__version__}, torchvision {torchvision.__version__}")

## 2. 라벨 (v1)

원본 JSON을 bbox 한 행 표로 읽고, 통합 보정 기록에서 v1 기록(`kwon_added`, `kwon_fixed`,
`jjam_capsule`)만 적용한다. 보정은 "지금 박스가 기록의 이전 좌표와 같을 때만" 적용하고, 하나라도
적용되지 않으면 멈춘다. 이미지 크기는 JSON 값 대신 실제 파일 헤더 값을 쓴다.
기대값: 라벨 771개, 클래스 56개, 이미지 밖·크기 0 이하·같은 사진 같은 약 중복 0.

In [ ]:
BBOX_COLS = ["x", "y", "w", "h"]
LABEL_PREFIXES = {
    "raw": (),
    "v1": ("kwon_added:", "kwon_fixed:", "jjam_capsule:"),
}
STAGES = ["추가", "위치 수정"]


def as_objects(value: object) -> list[dict]:
    """JSON의 단일 객체와 객체 목록을 같은 형태로 다룬다."""
    return [value] if isinstance(value, dict) else list(value)


def load_annotations(ann_dir: Path) -> pd.DataFrame:
    """annotation JSON을 모두 읽어 원본 bbox당 한 행인 표를 만든다.

    이미지 정보는 JSON 안의 image_id로 연결하고, (source_json, annotation_id)가 라벨 식별자다.
    """
    rows = []
    for json_path in sorted(ann_dir.rglob("*.json")):
        data = json.loads(json_path.read_text(encoding="utf-8-sig"))
        images = {img["id"]: img for img in as_objects(data["images"])}
        for ann in as_objects(data["annotations"]):
            img = images[ann["image_id"]]
            x, y, w, h = (float(v) for v in ann["bbox"])
            rows.append(
                {
                    "file_name": img["file_name"],
                    "source_json": json_path.relative_to(ann_dir).as_posix(),
                    "annotation_id": int(ann["id"]),
                    "category_id": int(ann["category_id"]),
                    "dl_name": img.get("dl_name"),
                    "x": x,
                    "y": y,
                    "w": w,
                    "h": h,
                    "has_segmentation": bool(ann.get("segmentation")),
                    # 15절 각인 OCR 정답: 이 사진에서 보이는 면과 앞·뒷면 각인
                    "drug_dir": img.get("drug_dir"),
                    "print_front": img.get("print_front"),
                    "print_back": img.get("print_back"),
                }
            )
    return pd.DataFrame(rows)


def read_image_sizes(img_dir: Path) -> pd.DataFrame:
    """이미지 헤더만 읽어 file_name, width, height 표를 만든다."""
    rows = []
    for path in sorted(p for p in img_dir.iterdir() if p.suffix.lower() in {".png", ".jpg"}):
        with Image.open(path) as img:
            rows.append({"file_name": path.name, "width": img.width, "height": img.height})
    return pd.DataFrame(rows)


def parse_bbox(text: str) -> tuple[int, int, int, int]:
    """보정 기록의 "[x, y, w, h]" 문자열을 정수 튜플로 바꾼다."""
    return tuple(int(round(float(v))) for v in text.strip("[]").split(","))


def apply_corrections(ann: pd.DataFrame, corrections: pd.DataFrame, version: str) -> pd.DataFrame:
    """라벨 버전의 보정 기록을 추가 → 위치 수정 순서로 적용한 복사본을 반환한다.

    기록이 하나라도 적용되지 않으면(미검수, 대상 없음, 현재 bbox ≠ old_bbox 등) 오류를 낸다.
    """
    df = ann.copy()
    df["label_origin"] = "원본"
    selected = corrections[corrections["correction_id"].str.startswith(LABEL_PREFIXES[version])]
    failed = []
    for stage in STAGES:
        for _, rec in selected[selected["stage"] == stage].iterrows():
            x, y, w, h = parse_bbox(rec["new_bbox"])
            if rec["reviewed"] is not True and str(rec["reviewed"]) != "True":
                failed.append((rec["correction_id"], "미검수"))
                continue
            if stage == "추가":
                same = (df["file_name"] == rec["file_name"]) & (
                    df["category_id"] == rec["category_id"]
                )
                if same.any():
                    failed.append((rec["correction_id"], "같은 사진에 같은 약 라벨이 이미 있음"))
                    continue
                same_drug = df.loc[df["category_id"] == rec["category_id"], "dl_name"]
                new_row = {
                    "file_name": rec["file_name"],
                    "source_json": None,
                    "annotation_id": int(rec["annotation_id"]),
                    "category_id": int(rec["category_id"]),
                    "dl_name": same_drug.iloc[0] if len(same_drug) else None,
                    "x": x,
                    "y": y,
                    "w": w,
                    "h": h,
                    "has_segmentation": False,
                    "label_origin": "추가",
                }
                df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
                continue
            target = (df["source_json"] == rec["source_json"]) & (
                df["annotation_id"] == rec["annotation_id"]
            )
            if target.sum() != 1:
                failed.append((rec["correction_id"], f"대상 라벨 {target.sum()}개"))
                continue
            current = tuple(int(round(v)) for v in df.loc[target, BBOX_COLS].iloc[0])
            if current != parse_bbox(rec["old_bbox"]):
                failed.append((rec["correction_id"], f"현재 bbox {current} ≠ old_bbox"))
                continue
            df.loc[target, BBOX_COLS] = [x, y, w, h]
    if failed:
        raise ValueError(f"{version}에서 적용되지 않은 보정 기록: {failed}")
    return df


def build_labels(version: str) -> pd.DataFrame:
    """원본 JSON에 보정 기록을 적용하고 실제 이미지 크기를 붙인 라벨 표를 만든다."""
    ann = load_annotations(ANN_DIR)
    corrections = pd.read_csv(PROCESSED_DIR / "corrections.csv")
    labels = apply_corrections(ann, corrections, version)
    sizes = read_image_sizes(TRAIN_IMG_DIR).set_index("file_name")
    labels["width"] = labels["file_name"].map(sizes["width"])
    labels["height"] = labels["file_name"].map(sizes["height"])
    labels = labels.sort_values(["file_name", "category_id", "annotation_id"])
    return labels.reset_index(drop=True)


def check_labels(labels: pd.DataFrame) -> dict[str, int]:
    """라벨의 bbox 문제 수를 센다. n_로 시작하지 않는 값은 모두 0이어야 학습에 쓸 수 있다."""
    x, y, w, h = (labels[col] for col in BBOX_COLS)
    return {
        "n_labels": len(labels),
        "n_images": labels["file_name"].nunique(),
        "n_classes": labels["category_id"].nunique(),
        "missing_size": int(labels[["width", "height"]].isna().any(axis=1).sum()),
        "out_of_image": int(
            ((x < 0) | (y < 0) | (x + w > labels["width"]) | (y + h > labels["height"])).sum()
        ),
        "non_positive": int(((w <= 0) | (h <= 0)).sum()),
        "duplicate_label": int(labels.duplicated(["file_name", "category_id"]).sum()),
    }


def build_class_map(labels: pd.DataFrame) -> pd.DataFrame:
    """category_id 오름차순으로 0부터 yolo_idx를 붙인 클래스 표(YOLO class_map.csv와 같음)."""
    first = labels.sort_values("category_id").drop_duplicates("category_id")
    class_map = first[["category_id", "dl_name"]].reset_index(drop=True)
    class_map.insert(0, "yolo_idx", range(len(class_map)))
    return class_map

In [ ]:
LABELS = build_labels(CFG["label_version"])
CLASS_MAP = build_class_map(LABELS)
CAT_TO_IDX = dict(zip(CLASS_MAP["category_id"], CLASS_MAP["yolo_idx"], strict=True))
IDX_TO_CAT = {i: c for c, i in CAT_TO_IDX.items()}
LABELS["yolo_idx"] = LABELS["category_id"].map(CAT_TO_IDX)
NUM_CLASSES = len(CLASS_MAP) + 1  # 0번은 배경, 라벨 id = yolo_idx + 1

label_checks = check_labels(LABELS)
print(label_checks)
problems = {k: v for k, v in label_checks.items() if not k.startswith("n_") and v}
if problems:
    raise ValueError(f"라벨 문제: {problems}")
print(f"num_classes(배경 포함): {NUM_CLASSES}")
print(LABELS["label_origin"].value_counts().to_dict())

# bbox가 없는 train 이미지: 분할은 라벨이 있는 사진만 대상으로 하므로(파이프라인과 같음) 학습·검증에
# 들어가지 않는다. 데이터셋은 빈 target(boxes 0×4)도 받도록 만들어 둔다.
all_train_images = set(read_image_sizes(TRAIN_IMG_DIR)["file_name"])
no_box_images = sorted(all_train_images - set(LABELS["file_name"]))
print(f"train 이미지 {len(all_train_images)}장 중 bbox 없는 사진 {len(no_box_images)}장")

## 3. Train/Val 분할

- 분할 단위는 조합(파일 이름의 `_` 앞부분)이다. 같은 조합의 사진은 train·val 한쪽에만 둔다.
- bbox를 표본, `category_id`를 층, 조합을 그룹으로 `StratifiedGroupKFold(5, shuffle, seed 42)`를 쓴다.
  `fix_rare=False`면 파이프라인의 `combo_v1_s42`와 같은 fold가 나온다(같은 라벨 순서·같은 seed).
- `fix_rare=True`면 v1 전체에서 bbox가 5개 이하인 약(희귀 약)이 하나라도 든 조합을 모든 fold에서
  train에 고정하고, 나머지 조합만 5 fold로 나눈다. 분할 이름은 `combofix_v1_s42`가 된다.

In [ ]:
RARE_MAX = 5


def split_name(cfg: dict) -> str:
    """분할 이름 <단위>[fix]_<라벨 버전>_s<seed>."""
    fix = "fix" if cfg["fix_rare"] else ""
    return f"{cfg['split_unit']}{fix}_{cfg['label_version']}_s{cfg['seed']}"


def rare_classes(labels: pd.DataFrame, rare_max: int = RARE_MAX) -> list[int]:
    """bbox가 rare_max개 이하인 category_id 목록."""
    counts = labels["category_id"].value_counts()
    return sorted(int(c) for c in counts[counts <= rare_max].index)


def make_folds(labels: pd.DataFrame, n_folds: int, seed: int, fix_rare: bool) -> pd.DataFrame:
    """fold × 사진마다 한 행인 표(file_name, combo, fold, split, fixed)를 만든다.

    labels가 file_name, category_id, annotation_id 순으로 정렬돼 있어야
    파이프라인과 같은 fold가 나온다.
    """
    units = pd.DataFrame({"file_name": sorted(labels["file_name"].unique())})
    units["combo"] = units["file_name"].str.split("_").str[0]
    boxes = labels[["file_name", "category_id"]].merge(units, on="file_name")
    fixed_combos = set()
    if fix_rare:
        fixed_combos = set(boxes.loc[boxes["category_id"].isin(rare_classes(labels)), "combo"])
    units["fixed"] = units["combo"].isin(fixed_combos)
    free = boxes[~boxes["combo"].isin(fixed_combos)].reset_index(drop=True)
    splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    with warnings.catch_warnings():
        # bbox가 n_folds개보다 적은 클래스가 있다는 경고. 희귀 약이 있어 당연히 나온다.
        warnings.simplefilter("ignore", UserWarning)
        splits = list(splitter.split(free, free["category_id"], free["combo"]))
    rows = []
    for fold, (_, val_idx) in enumerate(splits):
        val_files = set(free["file_name"].iloc[val_idx])
        is_val = units["file_name"].isin(val_files)
        rows.append(units.assign(fold=fold, split=np.where(is_val, "val", "train")))
    return pd.concat(rows, ignore_index=True)


def fold_report(folds: pd.DataFrame, labels: pd.DataFrame) -> pd.DataFrame:
    """fold마다 사진 수, 고정 사진 수, val bbox 수, 클래스 누락, 같은 조합 누수를 센다."""
    rows = []
    for fold, part in folds.groupby("fold"):
        train, val = part[part["split"] == "train"], part[part["split"] == "val"]
        train_cls = set(labels.loc[labels["file_name"].isin(train["file_name"]), "category_id"])
        val_labels = labels[labels["file_name"].isin(val["file_name"])]
        rows.append(
            {
                "fold": int(fold),
                "train_images": len(train),
                "val_images": len(val),
                "fixed_train_images": int(train["fixed"].sum()),
                "val_bboxes": len(val_labels),
                "n_classes_not_in_val": len(train_cls - set(val_labels["category_id"])),
                "val_only_classes": sorted(set(val_labels["category_id"]) - train_cls),
                "combo_leak": float(val["combo"].isin(set(train["combo"])).mean()),
            }
        )
    return pd.DataFrame(rows)

In [ ]:
SPLIT_NAME = split_name(CFG)
FOLDS = make_folds(LABELS, CFG["n_folds"], CFG["seed"], CFG["fix_rare"])
FOLD_REPORT = fold_report(FOLDS, LABELS)
if (FOLD_REPORT["combo_leak"] > 0).any():
    raise ValueError(f"같은 조합 누수가 있습니다:\n{FOLD_REPORT}")
print(f"분할: {SPLIT_NAME}")
print(f"희귀 약(bbox ≤ {RARE_MAX}): {rare_classes(LABELS)}")
FOLD_REPORT

## 4. 데이터셋

라벨 표의 픽셀 COCO 좌표 `[x, y, w, h]`를 데이터셋 안에서 `[x1, y1, x2, y2]`로 바꾼다(파일로 저장하지
않음). 라벨 id는 `yolo_idx + 1`이다(0은 배경). bbox가 없는 사진은 빈 target으로 돌려준다.

In [ ]:
class PillDataset(Dataset):
    """알약 사진과 Faster R-CNN target(boxes xyxy, labels = yolo_idx + 1)을 돌려주는 데이터셋."""

    def __init__(
        self,
        img_dir: Path,
        file_names: list[str],
        labels: pd.DataFrame | None = None,
        hflip_p: float = 0.0,
    ) -> None:
        self.img_dir = img_dir
        self.file_names = list(file_names)
        self.hflip_p = hflip_p
        self.boxes = {} if labels is None else dict(tuple(labels.groupby("file_name")))

    def __len__(self) -> int:
        return len(self.file_names)

    def __getitem__(self, i: int) -> tuple[torch.Tensor, dict[str, torch.Tensor], str]:
        name = self.file_names[i]
        with Image.open(self.img_dir / name) as img:
            image = TF.to_tensor(img.convert("RGB"))
        rows = self.boxes.get(name)
        if rows is None:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            classes = torch.zeros((0,), dtype=torch.int64)
        else:
            x, y, w, h = (torch.tensor(rows[c].to_numpy(), dtype=torch.float32) for c in BBOX_COLS)
            boxes = torch.stack([x, y, x + w, y + h], dim=1)
            classes = torch.tensor(rows["yolo_idx"].to_numpy() + 1, dtype=torch.int64)
        if self.hflip_p > 0 and random.random() < self.hflip_p:
            image = image.flip(-1)
            width = image.shape[-1]
            boxes = torch.stack(
                [width - boxes[:, 2], boxes[:, 1], width - boxes[:, 0], boxes[:, 3]], dim=1
            )
        return image, {"boxes": boxes, "labels": classes}, name


def collate(batch: list) -> tuple:
    """검출 모델은 사진 크기가 달라도 되므로 목록 그대로 묶는다."""
    return tuple(zip(*batch, strict=True))

### 4-1. AI Hub 외부 데이터 추가 (02_eda §7, 외부 전용 약은 새 클래스)

`exp/yolo11m_100_aihub_not_background` 노트북 5-2단계와 같은 방법으로 외부 사진을 가져와 모든 fold의 train에만 더한다.
val·test는 대회 사진 그대로다. YOLO 노트북과 달리 YOLO txt·`data.yaml`은 만들지 않고 데이터셋이 표를 바로 읽는다.

1. **가져오기**: 이미 푼 폴더가 있으면 `extract_dir`에 적는다(권장. Kaggle은 푼 AI Hub 폴더를 입력 데이터셋으로
   올리고 `/kaggle/input/<데이터셋>` 경로를 적으면 받기·풀기를 건너뛴다). 비워 두면 Kaggle·Colab은
   `AIHUB_URL`(AI Hub zip 폴더의 Drive 공유 링크)에서 zip을 받아 푼다. 이미 받았거나 다 푼 zip은 건너뛴다.
   Kaggle은 zip과 푼 파일(약 48GB)을 20GB 한도인 `/kaggle/working` 대신 `/tmp/pill-bbox-external`에 둔다.
   `AIHUB_URL`은 환경 변수 → `.env` → Kaggle Secrets·Colab 보안 비밀 → 입력창 순서로 찾는다.
   zip을 이미 디스크에 두었다면 `zip_dir`를 지정한다. 로컬은 02_eda처럼 미리 푼 폴더
   (`.env`의 `AIHUB_EXTRACT_DIR`, 없으면 `data/external/aihub_combo`)만 쓴다. `exclude`의 zip(`TL_2`, `TS_2`)은 쓰지 않는다.
2. **선별 (02_eda §5~7)**: 대회 v1 라벨과 공통인 클래스마다 `target_bbox`(157)개가 되도록 부족한 수만큼 seed 42로
   bbox를 뽑는다. 좌표 오류·같은 약 중복 라벨이 있는 사진과 대회 사진과 이름이 같은 사진은 후보에서 뺀다.
3. **문제 사진 제외 (02_eda §8)**: 파일명의 약 코드와 라벨이 다르거나(라벨 누락·파일명에 없는 라벨), bbox끼리
   겹치거나(IoU > 0), 좌표가 잘못됐거나, JSON의 사진 크기와 실제 크기가 다르거나, 대회 사진과 약 조합이 같은
   사진(val 누수 가능)은 뺀다. 02_eda §10~11의 OpenCV 보정은 사람 검수 전이라 적용하지 않는다.
4. **라벨**: 사진을 통째로 쓰므로 선택하지 않은 대회 알약도 모두 라벨링한다. 그래서 클래스별 bbox는 157개를 넘을 수 있다.
   대회 56종에 없는 **외부 전용 알약도 배경이 아닌 새 클래스로 라벨링**한다(`extra_as_class = True`).
   대회 번호(yolo_idx 0~55) 뒤에 이어지는 번호를 주고, `False`면 라벨 없이 배경으로 둔다.

In [ ]:
import hashlib
from getpass import getpass

from torch.utils.data import ConcatDataset

# Colab 커널의 작업 폴더가 달라도 저장소의 data_process를 import하도록 연결한다.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_process.aihub_data import (  # noqa: E402
    download_drive_folder,
    extract_zips,
    find_combo_zips,
    list_combo_images,
    load_combo_annotations,
    zip_key,
)
from data_process.data_processing import (  # noqa: E402
    bbox_flags,
    find_box_pairs,
    list_images,
    read_image_headers,
)

CFG["external_data"] = {
    "enabled": True,
    "target_bbox": 157,  # 02_eda §5: 공통 클래스당 대회 + 외부 bbox 보충 목표
    "extra_as_class": True,  # 외부 전용 약을 새 클래스로 학습. False면 라벨 없이 배경
    "submit_extra": True,  # 외부 전용 클래스 예측도 약 코드로 제출(B안). False면 56종만(A안)
    "exclude": ["TL_2", "TS_2"],  # 쓰지 않는 zip·폴더 구분자 (data/README.md)
    # 이미 푼 폴더(예: Kaggle 입력 데이터셋 "/kaggle/input/<데이터셋>"). 적으면 받기·풀기 생략.
    # None이면 Kaggle·Colab은 AIHUB_URL에서 받아 풀고, 로컬은 .env의 AIHUB_EXTRACT_DIR를 쓴다.
    "extract_dir": None,
    "zip_dir": None,  # Kaggle·Colab에 이미 둔 zip 폴더. None이면 AIHUB_URL의 Drive 폴더를 받는다
}
EXTERNAL_NUMBERS = [*BBOX_COLS, "width", "height"]
EXTERNAL_DATA_CACHE = {}


def external_enabled(cfg: dict) -> bool:
    """외부 데이터 추가 여부를 반환한다."""
    return bool(cfg.get("external_data", {}).get("enabled", False))


def external_codes(file_name: str) -> set[int]:
    """조합 파일명(K-001900-003544-..._...)의 약 코드 집합. 사진에서 센 알약 수가 아니다."""
    prefix = Path(file_name).stem.split("_")[0]
    if not prefix.startswith("K-"):
        return set()
    try:
        return {int(code) for code in prefix.split("-")[1:]}
    except ValueError:
        return set()


def aihub_url() -> str:
    """AIHUB_URL을 환경 변수 → .env → Kaggle·Colab 비밀 → 입력창 순서로 찾는다(값은 출력 안 함)."""
    with contextlib.suppress(ImportError):  # Kaggle 기본 환경에는 python-dotenv가 없을 수 있다
        from dotenv import load_dotenv

        load_dotenv("/content/.env")
        load_dotenv(PROJECT_ROOT / ".env")
    if not os.getenv("AIHUB_URL") and IN_KAGGLE:
        with contextlib.suppress(Exception):
            from kaggle_secrets import UserSecretsClient

            os.environ["AIHUB_URL"] = UserSecretsClient().get_secret("AIHUB_URL") or ""
    if not os.getenv("AIHUB_URL") and IN_COLAB:
        with contextlib.suppress(Exception):
            from google.colab import userdata

            os.environ["AIHUB_URL"] = userdata.get("AIHUB_URL") or ""
    if not os.getenv("AIHUB_URL"):
        os.environ["AIHUB_URL"] = getpass(
            "AI Hub zip 폴더의 Drive 링크 AIHUB_URL (화면에 표시되지 않음): "
        ).strip()
    if not os.environ["AIHUB_URL"]:
        raise RuntimeError("AIHUB_URL이 필요합니다. .env 또는 Kaggle·Colab 비밀에 등록하세요.")
    return os.environ["AIHUB_URL"]


def external_work_dir() -> Path:
    """받은 zip과 푼 파일을 둘 폴더. Kaggle은 20GB 한도인 /kaggle/working 밖에 둔다."""
    return Path("/tmp/pill-bbox-external") if IN_KAGGLE else PROJECT_ROOT / "data" / "external"


def aihub_extract_dir(cfg: dict) -> Path:
    """압축을 푼 외부 데이터 폴더. Kaggle·Colab은 .env에 로컬 경로가 있어도 런타임 디스크를 쓴다."""
    configured = cfg["external_data"]["extract_dir"]
    if configured is None and not (IN_KAGGLE or IN_COLAB):
        configured = os.getenv("AIHUB_EXTRACT_DIR")
    path = Path(configured).expanduser() if configured else external_work_dir() / "aihub_combo"
    return path if path.is_absolute() else PROJECT_ROOT / path


def prepare_aihub_folders(cfg: dict) -> list[Path]:
    """다 푼 폴더 중 exclude에 없는 폴더를 이름순으로 반환한다 (02_eda §1).

    extract_dir를 적지 않은 Kaggle·Colab은 먼저 AIHUB_URL에서 zip을 받아 푼다.
    """
    options = cfg["external_data"]
    extract_dir = aihub_extract_dir(cfg)
    if (IN_KAGGLE or IN_COLAB) and options["extract_dir"] is None:
        if options["zip_dir"]:
            zip_dir = Path(options["zip_dir"])
        else:
            # 이미 받은 파일은 건너뛰고 끊긴 파일은 이어 받는다
            zip_dir = download_drive_folder(aihub_url(), external_work_dir() / "aihub_zips")
        zips = find_combo_zips(zip_dir, exclude=options["exclude"])
        if not zips:
            raise FileNotFoundError(f"경로에 '조합'이 들어간 zip이 없습니다: {zip_dir}")
        print(f"조합 zip {len(zips)}개: {', '.join(path.name for path in zips)}")
        extract_zips(zips, extract_dir)  # 이미 다 푼 zip은 건너뛴다
    folders = []
    if extract_dir.is_dir():
        folders = sorted(
            folder
            for folder in extract_dir.iterdir()
            if folder.is_dir()
            and not folder.name.startswith(".")
            and zip_key(folder) not in options["exclude"]
        )
    if not folders:
        raise FileNotFoundError(
            f"압축을 푼 외부 데이터 폴더가 없습니다: {extract_dir}. "
            "로컬은 02_eda처럼 미리 풀어 두고, "
            "Kaggle·Colab은 extract_dir(입력 데이터셋 경로) 또는 AIHUB_URL을 확인하세요."
        )
    return folders


def select_aihub_images(dirs: list[Path], cfg: dict) -> dict:
    """02_eda §1·2·5·6·7과 같이 bbox를 고르고 §8 검사로 쓸 사진·라벨·외부 전용 알약을 나눈다."""
    target = int(cfg["external_data"]["target_bbox"])
    # §1·2: 이미지와 라벨 연결. 같은 이름이 여러 폴더에 있으면 라벨을 안전하게 붙일 수 없다.
    image_index = list_combo_images(dirs)
    if image_index.empty:
        raise FileNotFoundError("외부 이미지가 없습니다. TS 원천 이미지 폴더를 확인하세요.")
    if image_index["file_name"].duplicated().any():
        raise ValueError("외부 이미지 이름이 중복됩니다. 02_eda §1에서 중복 목록을 확인하세요.")
    ann_df, _ = load_combo_annotations(dirs)
    if ann_df.empty or ann_df["category_id"].isna().any():
        raise ValueError("외부 라벨이 없거나 약 코드를 해석할 수 없는 bbox가 있습니다 (02_eda §2).")
    available = ann_df[ann_df["file_name"].isin(set(image_index["file_name"]))].copy()

    # §5: 대회 v1과 공통인 클래스마다 목표까지 부족한 bbox 수
    comp_counts = LABELS.groupby("category_id").size()
    common = sorted(set(comp_counts.index) & set(available["category_id"].astype(int)))
    needed = (target - comp_counts).clip(lower=0)

    # §6: 좌표·중복 오류가 있는 사진과 대회 사진과 같은 이름은 후보에서 뺀다
    candidates = available[available["category_id"].isin(common)].merge(
        image_index[["file_name", "path"]], on="file_name", validate="many_to_one"
    )
    numeric = available[EXTERNAL_NUMBERS].apply(pd.to_numeric, errors="coerce")
    flags = bbox_flags(available)
    invalid_box = (
        ~np.isfinite(numeric).all(axis=1)
        | numeric[["width", "height"]].le(0).any(axis=1)
        | flags["out_of_image"]
        | flags["non_positive"]
        | available.duplicated(["file_name", "category_id"], keep=False)
    )
    invalid_files = set(available.loc[invalid_box, "file_name"])
    comp_names = {
        path.name for folder in (TRAIN_IMG_DIR, TEST_IMG_DIR) for path in list_images(folder)
    }
    comp_combos = set(LABELS["file_name"].str.split("_").str[0])
    comp_combos |= {name.split("_")[0] for name in comp_names if name.startswith("K-")}

    # §7: 클래스별 부족량만큼 seed로 비복원 추출한다
    pool = candidates[~candidates["file_name"].isin(invalid_files | comp_names)].copy()
    pool = pool.drop_duplicates(["file_name", "category_id", *BBOX_COLS])
    shuffled = pool.sample(frac=1, random_state=cfg["seed"])
    order = shuffled.groupby("category_id").cumcount()
    selected = (
        shuffled[order.lt(shuffled["category_id"].map(needed))]
        .sort_values(["category_id", "file_name"])
        .reset_index(drop=True)
    )

    # §8: 선택 bbox가 든 사진 전체의 라벨을 검사한다. 사진을 통째로 쓰므로 함께 찍힌 bbox도 본다.
    files = sorted(selected["file_name"].unique())
    boxes = available[available["file_name"].isin(files)].copy()
    boxes["category_id"] = boxes["category_id"].astype(int)
    numeric = boxes[EXTERNAL_NUMBERS].apply(pd.to_numeric, errors="coerce")
    valid = np.isfinite(numeric).all(axis=1) & numeric[["w", "h", "width", "height"]].gt(0).all(
        axis=1
    )
    invalid = (
        ~valid
        | bbox_flags(boxes)["out_of_image"]
        | boxes.duplicated(["file_name", "category_id"], keep=False)
    )
    pairs = find_box_pairs(boxes[valid], max_gap=0)
    labeled = boxes.groupby("file_name")["category_id"].agg(set)
    paths = image_index.set_index("file_name")["path"]
    headers = read_image_headers([paths[name] for name in files], "aihub").set_index("file_name")
    json_size = boxes.groupby("file_name")[["width", "height"]].first()
    reasons = pd.DataFrame(index=pd.Index(files, name="file_name"))
    reasons["라벨 누락"] = [bool(external_codes(name) - labeled[name]) for name in files]
    reasons["파일명에 없는 라벨"] = [bool(labeled[name] - external_codes(name)) for name in files]
    reasons["bbox 겹침"] = reasons.index.isin(set(pairs.loc[pairs["iou"].gt(0), "file_name"]))
    reasons["좌표·중복 오류"] = reasons.index.isin(set(boxes.loc[invalid, "file_name"]))
    real_size = headers.loc[files, ["width", "height"]].to_numpy()
    reasons["JSON·실제 크기 다름"] = (real_size != json_size.loc[files].to_numpy()).any(axis=1)
    # 대회 사진과 약 조합이 같으면 같은 조합의 대회 val 사진과 겹칠 수 있다
    reasons["대회와 같은 조합"] = reasons.index.str.split("_").str[0].isin(comp_combos)

    comp_ids = set(CLASS_MAP["category_id"].astype(int))
    kept = boxes[boxes["file_name"].isin(reasons.index[~reasons.any(axis=1)])]
    return {
        "n_images": len(image_index),
        "n_labels": len(ann_df),
        "common_classes": common,
        "selected": selected,
        "reasons": reasons,
        "labels": kept[kept["category_id"].isin(comp_ids)].copy(),
        "extra": kept[~kept["category_id"].isin(comp_ids)].copy(),
        "paths": paths,
    }


def load_external_bundle(cfg: dict) -> dict:
    """외부 데이터를 가져와 02_eda 기준으로 고르고 학습 라벨(yolo_idx 포함)·사진 경로를 묶는다."""
    options = cfg["external_data"]
    folders = prepare_aihub_folders(cfg)
    print(f"외부 데이터 {aihub_extract_dir(cfg)}: {', '.join(folder.name for folder in folders)}")
    aihub = select_aihub_images(folders, cfg)
    reasons, labels, extra = aihub["reasons"], aihub["labels"], aihub["extra"]
    files = sorted(reasons.index[~reasons.any(axis=1)])
    if sorted(labels["file_name"].unique()) != files:
        raise ValueError("대회 클래스 라벨이 없는 외부 사진이 있습니다.")

    # 외부 전용 알약: 배경이 아니라 대회 클래스 수 뒤의 새 번호로 라벨링한다 (끄면 라벨 없이 배경)
    extra_map = pd.DataFrame(
        {
            "yolo_idx": pd.Series(dtype=int),
            "category_id": pd.Series(dtype=int),
            "dl_name": pd.Series(dtype=str),
        }
    )
    if options["extra_as_class"]:
        first = extra.drop_duplicates("category_id").sort_values("category_id")
        extra_map = pd.DataFrame(
            {
                "yolo_idx": range(len(CLASS_MAP), len(CLASS_MAP) + len(first)),
                "category_id": first["category_id"].astype(int).to_numpy(),
                "dl_name": first["dl_name"].to_numpy(),
            }
        )
        labels = pd.concat([labels, extra], ignore_index=True)
    cat_to_idx = {
        **CAT_TO_IDX,
        **dict(zip(extra_map["category_id"], extra_map["yolo_idx"], strict=True)),
    }
    labels = labels.copy()
    labels[EXTERNAL_NUMBERS] = labels[EXTERNAL_NUMBERS].apply(pd.to_numeric)
    labels["yolo_idx"] = labels["category_id"].map(cat_to_idx)
    if labels["yolo_idx"].isna().any():
        raise ValueError("모델 클래스 번호를 붙일 수 없는 외부 라벨이 있습니다.")
    labels["yolo_idx"] = labels["yolo_idx"].astype(int)
    labels = labels.sort_values(["file_name", "category_id"]).reset_index(drop=True)
    paths = {name: Path(aihub["paths"][name]) for name in files}

    digest = hashlib.sha256(b"aihub_not_background_v1")
    digest.update(
        json.dumps(
            {
                "folders": [folder.name for folder in folders],
                "target_bbox": options["target_bbox"],
                "extra_as_class": options["extra_as_class"],
                "seed": cfg["seed"],
            },
            sort_keys=True,
        ).encode()
    )
    digest.update(labels[["file_name", "yolo_idx", *BBOX_COLS]].to_csv(index=False).encode())

    balance = CLASS_MAP.set_index("category_id")[["dl_name"]].copy()
    balance["대회 v1"] = LABELS.groupby("category_id").size()
    balance["02_eda 선택"] = aihub["selected"].groupby("category_id").size()
    balance["외부 학습 라벨"] = labels.groupby("category_id").size()
    count_columns = ["대회 v1", "02_eda 선택", "외부 학습 라벨"]
    balance[count_columns] = balance[count_columns].fillna(0).astype(int)
    balance["합계"] = balance["대회 v1"] + balance["외부 학습 라벨"]
    summary = [
        ("외부 이미지 (푼 폴더 전체)", aihub["n_images"]),
        ("외부 bbox 라벨 (푼 폴더 전체)", aihub["n_labels"]),
        ("공통 클래스 (대회 v1 ∩ 외부)", len(aihub["common_classes"])),
        ("02_eda §7 선택 bbox", len(aihub["selected"])),
        ("선택 bbox가 든 사진", len(reasons)),
        *[(f"  제외: {column}", int(reasons[column].sum())) for column in reasons],
        ("제외한 사진 (사유가 여럿이어도 한 번)", int(reasons.any(axis=1).sum())),
        ("train에 더할 사진", len(files)),
        ("학습 라벨 bbox (대회 클래스, 선택 + 함께 찍힌 것)", len(aihub["labels"])),
        ("외부 전용 알약 bbox (새 클래스 또는 배경)", len(extra)),
        ("외부 전용 클래스 수 (새 클래스로 학습할 때)", len(extra_map)),
    ]
    return {
        "labels": labels,
        "paths": paths,
        "extra_map": extra_map,
        "extra_boxes": extra,
        "balance": balance.sort_values("대회 v1"),
        "summary": summary,
        "fingerprint": digest.hexdigest(),
        "folders": [folder.name for folder in folders],
    }


def get_external_bundle(cfg: dict) -> dict:
    """같은 외부 설정은 현재 커널에서 한 번만 읽는다."""
    key = json.dumps(cfg["external_data"], sort_keys=True, ensure_ascii=False) + str(cfg["seed"])
    if key not in EXTERNAL_DATA_CACHE:
        bundle = load_external_bundle(cfg)
        # 예측의 yolo_idx → category_id 표에 외부 전용 클래스를 더한다(대회 번호 뒤라 겹치지 않음).
        extra_map = bundle["extra_map"]
        IDX_TO_CAT.update(zip(extra_map["yolo_idx"], extra_map["category_id"], strict=True))
        EXTERNAL_DATA_CACHE[key] = bundle
    return EXTERNAL_DATA_CACHE[key]


def model_class_map(cfg: dict) -> pd.DataFrame:
    """모델 클래스 표(yolo_idx, category_id, dl_name): 대회 56종 + 새 클래스로 학습한 외부 약."""
    if not external_enabled(cfg) or not cfg["external_data"]["extra_as_class"]:
        return CLASS_MAP
    return pd.concat([CLASS_MAP, get_external_bundle(cfg)["extra_map"]], ignore_index=True)


def model_num_classes(cfg: dict) -> int:
    """모델 분류 헤드의 클래스 수. 0번은 배경이고 라벨 id = yolo_idx + 1이다."""
    return len(model_class_map(cfg)) + 1


def competition_pred(pred: pd.DataFrame) -> pd.DataFrame:
    """예측에서 대회 56종만 남긴다. val 정답에는 외부 전용 약이 없으므로 평가는 이 표로 한다."""
    return pred[pred["yolo_idx"] < len(CLASS_MAP)]


class ExternalPillDataset(Dataset):
    """외부 사진 전체와 학습 target(대회 클래스 + 새 클래스로 학습하는 외부 전용 약)을 반환한다."""

    def __init__(self, names: list[str], bundle: dict, hflip_p: float = 0.0) -> None:
        self.names = names
        self.paths = bundle["paths"]
        self.boxes = dict(tuple(bundle["labels"].groupby("file_name")))
        self.hflip_p = hflip_p

    def __len__(self) -> int:
        return len(self.names)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, dict[str, torch.Tensor], str]:
        name = self.names[index]
        with Image.open(self.paths[name]) as image:
            tensor = TF.to_tensor(image.convert("RGB"))
        rows = self.boxes[name]
        values = torch.tensor(rows[BBOX_COLS].to_numpy(), dtype=torch.float32)
        boxes = values.clone()
        boxes[:, 2:] += boxes[:, :2]
        classes = torch.tensor(rows["yolo_idx"].to_numpy() + 1, dtype=torch.int64)
        if self.hflip_p > 0 and random.random() < self.hflip_p:
            tensor = tensor.flip(-1)
            width = tensor.shape[-1]
            boxes = torch.stack(
                [width - boxes[:, 2], boxes[:, 1], width - boxes[:, 0], boxes[:, 3]], dim=1
            )
        return tensor, {"boxes": boxes, "labels": classes}, f"external::{name}"


def training_data_spec(cfg: dict, fold: int) -> tuple[list[str], list[str], dict | None]:
    """대회 분할을 유지하고 외부 데이터에는 동일한 fraction만 적용한다."""
    base_names = fold_files(fold, "train")
    if cfg["fraction"] < 1:
        rng = random.Random(cfg["seed"])
        base_names = sorted(
            rng.sample(base_names, max(1, round(len(base_names) * cfg["fraction"])))
        )
    if not external_enabled(cfg):
        return base_names, [], None
    bundle = get_external_bundle(cfg)
    names = sorted(bundle["paths"])
    val_combos = {tuple(sorted(external_codes(name))) for name in fold_files(fold, "val")}
    if any(tuple(sorted(external_codes(name))) in val_combos for name in names):
        raise ValueError("외부 학습 이미지에 현재 검증 조합이 포함됐습니다.")
    if cfg["fraction"] < 1:
        rng = random.Random(cfg["seed"])
        names = sorted(rng.sample(names, max(1, round(len(names) * cfg["fraction"]))))
    return base_names, names, bundle


def training_counts(cfg: dict, fold: int) -> pd.Series:
    """클래스별 실제 대회·외부 학습 bbox 수를 반환한다."""
    base_names, names, bundle = training_data_spec(cfg, fold)
    counts = LABELS.loc[LABELS["file_name"].isin(base_names), "category_id"].value_counts()
    if bundle is not None:
        external_counts = (
            bundle["labels"]
            .loc[bundle["labels"]["file_name"].isin(names), "category_id"]
            .value_counts()
        )
        counts = counts.add(external_counts, fill_value=0)
    return counts.astype(int)


if external_enabled(CFG):
    external_preview = get_external_bundle(CFG)
    display(pd.DataFrame(external_preview["summary"], columns=["항목", "개수"]))
    n_extra = len(external_preview["extra_map"])
    print(
        f"모델 클래스: 대회 {len(CLASS_MAP)}종 + 외부 전용 {n_extra}종 + 배경 "
        f"= {model_num_classes(CFG)}"
    )
    print(
        "외부 전용 알약은 새 클래스로 학습합니다."
        if CFG["external_data"]["extra_as_class"]
        else "외부 전용 알약은 사진에 그대로 두고 라벨 없이 배경으로 둡니다."
    )
    target_bbox = CFG["external_data"]["target_bbox"]
    n_short = int(external_preview["balance"]["합계"].lt(target_bbox).sum())
    print(f"합계가 목표 {target_bbox}개 미만인 클래스: {n_short}/{len(CLASS_MAP)}종")
    print("외부 사진은 train에만 더합니다. val·test는 대회 사진만 씁니다.")
    with pd.option_context("display.max_rows", None):
        display(external_preview["balance"])

#### 외부 사진 눈으로 확인

외부 train 사진 4장(외부 전용 알약이 든 사진 먼저)에 학습 라벨을 그린다. 초록은 대회 클래스, 주황은 새 클래스로
학습하는 외부 전용 알약이다. `extra_as_class = False`면 외부 전용 알약은 회색 `bg`로만 표시하고 학습에 들어가지 않는다.

In [ ]:
if external_enabled(CFG):
    preview_labels = external_preview["labels"]
    preview_extra = external_preview["extra_boxes"]
    preview_pool = sorted(set(preview_extra["file_name"])) or sorted(external_preview["paths"])
    preview_files = pd.Series(preview_pool).sample(
        n=min(4, len(preview_pool)), random_state=CFG["seed"]
    )
    fig, axes = plt.subplots(
        1, len(preview_files), figsize=(4 * len(preview_files), 5), squeeze=False
    )
    for ax, file_name in zip(axes[0], preview_files, strict=True):
        with Image.open(external_preview["paths"][file_name]) as img:
            ax.imshow(img.convert("RGB"))
        for row in preview_labels[preview_labels["file_name"] == file_name].itertuples():
            color = "lime" if row.yolo_idx < len(CLASS_MAP) else "orange"
            ax.add_patch(Rectangle((row.x, row.y), row.w, row.h, fill=False, ec=color, lw=2))
            ax.text(row.x, row.y - 5, str(row.category_id), color=color, fontsize=8)
        if not CFG["external_data"]["extra_as_class"]:
            for row in preview_extra[preview_extra["file_name"] == file_name].itertuples():
                ax.add_patch(Rectangle((row.x, row.y), row.w, row.h, fill=False, ec="gray", lw=2))
                ax.text(row.x, row.y - 5, f"bg {row.category_id}", color="gray", fontsize=8)
        ax.set_title(file_name.split("_")[0], fontsize=8)
        ax.axis("off")
    fig.suptitle("AI Hub train: 초록 = 대회 클래스, 주황 = 외부 전용 새 클래스, 회색 = 배경")
    fig.tight_layout()
    plt.show()

## 5. 모델

`fasterrcnn_resnet50_fpn_v2` + COCO_V1 가중치에서 분류 헤드(`FastRCNNPredictor`)만
대회 56종 + 외부 전용 클래스 + 배경(`model_num_classes(cfg)`)으로 바꾼다.
`trainable_backbone_layers=5`로 백본까지 전체 파인튜닝한다. 추론 조건은 YOLO 평가와 맞춰
score 임계값 0.001, 사진당 최대 300개로 둔다.

In [ ]:
def build_model(cfg: dict, num_classes: int | None = None) -> torch.nn.Module:
    """사전학습 Faster R-CNN의 분류 헤드를 num_classes(배경 포함)로 바꾼 모델을 만든다.

    num_classes가 없으면 model_num_classes(cfg)(대회 56종 + 외부 전용 클래스 + 배경)를 쓴다.
    """
    if cfg["pretrained"] == "coco":
        weights, weights_backbone = FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1, None
    elif cfg["pretrained"] == "imagenet":
        weights, weights_backbone = None, ResNet50_Weights.IMAGENET1K_V1
    else:  # 저장한 가중치를 읽을 때. v2는 사전학습 여부와 관계없이 구조가 같다
        weights, weights_backbone = None, None
    is_trained = weights is not None or weights_backbone is not None
    model = fasterrcnn_resnet50_fpn_v2(
        weights=weights,
        weights_backbone=weights_backbone,
        num_classes=None if weights else 91,
        trainable_backbone_layers=cfg["trainable_backbone_layers"] if is_trained else None,
        min_size=cfg["min_size"],
        max_size=cfg["max_size"],
        box_score_thresh=cfg["conf"],
        box_nms_thresh=cfg["nms_iou"],
        box_detections_per_img=cfg["max_det"],
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(
        in_features, num_classes or model_num_classes(cfg)
    )
    return model

## 6. 평가 지표

YOLO 쪽 숫자와 나란히 놓을 수 있도록 ultralytics 8.4 검증과 같은 계산을 옮겼다.

- 짝짓기(`match_predictions`): 사진마다 신뢰도 높은 예측부터, IoU 기준(0.50–0.95, 10단계)마다
  클래스가 같고 아직 짝이 없는 정답 중 IoU가 가장 큰 것과 짝짓는다.
- AP(`compute_ap`): precision 포락선을 101점 보간해 사다리꼴로 적분한다.
- 평균은 val에 정답이 있는 클래스 기준이다(예측이 없는 클래스는 AP 0으로 들어간다).
- 주 지표 mAP75-95 = IoU 0.75–0.95(5단계) AP 평균의 클래스 평균(`models/evaluate.py`의 `MAIN_METRIC`).
- 정밀도·재현율·F1·평균 IoU는 `evaluate_fold`와 같은 고정 기준(신뢰도 0.5 이상, IoU 0.5 이상 짝)으로 센다.
- 혼동행렬은 ultralytics `ConfusionMatrix`와 같이 IoU 0.45로 짝짓고, 행 = 예측, 열 = 정답, 마지막은 배경.

In [ ]:
MAIN_METRIC = "map75_95"
IOU_THRESHOLDS = np.linspace(0.5, 0.95, 10)
IOU75_COL = 5
SIZE_GROUPS = [(21, "21+"), (6, "6-20"), (0, "≤5")]
CONFUSION_PAIRS = [(16262, 20238), (16232, 21325), (18357, 32310)]
POINT_CONF = 0.5
POINT_IOU = 0.5
CM_IOU = 0.45
PRED_COLS = ["file_name", "yolo_idx", "category_id", *BBOX_COLS, "conf"]
SPEED_KEYS = ["preprocess", "inference", "postprocess"]


def xywh_to_xyxy(boxes: np.ndarray) -> np.ndarray:
    """[x, y, w, h] 배열을 [x1, y1, x2, y2]로 바꾼다."""
    return np.concatenate([boxes[:, :2], boxes[:, :2] + boxes[:, 2:]], axis=1)


def box_iou(a: np.ndarray, b: np.ndarray, eps: float = 1e-7) -> np.ndarray:
    """xyxy 박스 a(n개)와 b(m개)의 IoU 표(n × m). ultralytics box_iou와 같은 eps를 쓴다."""
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = (rb - lt).clip(0).prod(2)
    area_a = (a[:, 2:] - a[:, :2]).prod(1)
    area_b = (b[:, 2:] - b[:, :2]).prod(1)
    return inter / (area_a[:, None] + area_b[None, :] - inter + eps)


def match_predictions(pred_cls: np.ndarray, gt_cls: np.ndarray, iou: np.ndarray) -> np.ndarray:
    """예측(신뢰도 내림차순) × IoU 10단계의 정답 여부 표를 만든다. iou는 정답 × 예측이다."""
    correct = np.zeros((len(pred_cls), len(IOU_THRESHOLDS)), dtype=bool)
    iou = iou * (gt_cls[:, None] == pred_cls[None, :])
    matched = np.zeros((iou.shape[0], len(IOU_THRESHOLDS)), dtype=bool)
    cols = range(len(IOU_THRESHOLDS))
    for j in np.flatnonzero((iou >= IOU_THRESHOLDS.min()).any(0)):
        available = np.where(matched, 0, iou[:, j, None])
        k = available.argmax(0)
        correct[j] = available[k, cols] >= IOU_THRESHOLDS
        matched[k, cols] |= correct[j]
    return correct


def compute_ap(recall: np.ndarray, precision: np.ndarray) -> float:
    """recall·precision 곡선의 AP(101점 보간, ultralytics compute_ap와 같음)."""
    mrec = np.concatenate(([0.0], recall, [recall[-1] if len(recall) else 1.0], [1.0]))
    mpre = np.concatenate(([1.0], precision, [0.0], [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101)
    return float(np.trapezoid(np.interp(x, mrec, mpre), x))


def smooth(y: np.ndarray, f: float = 0.05) -> np.ndarray:
    """상자 필터로 1차원 배열을 부드럽게 만든다(ultralytics smooth)."""
    nf = round(len(y) * f * 2) // 2 + 1
    p = np.ones(nf // 2)
    yp = np.concatenate((p * y[0], y, p * y[-1]), 0)
    return np.convolve(yp, np.ones(nf) / nf, mode="valid")


def ap_per_class(
    tp: np.ndarray, conf: np.ndarray, pred_cls: np.ndarray, target_cls: np.ndarray, eps=1e-16
) -> dict[str, np.ndarray]:
    """클래스 × IoU 10단계 AP와, 평균 F1이 가장 큰 신뢰도에서의 클래스별 P·R을 구한다."""
    order = np.argsort(-conf)
    tp, conf, pred_cls = tp[order], conf[order], pred_cls[order]
    classes, n_targets = np.unique(target_cls, return_counts=True)
    x = np.linspace(0, 1, 1000)
    ap = np.zeros((len(classes), tp.shape[1]))
    p_curve, r_curve = np.zeros((len(classes), 1000)), np.zeros((len(classes), 1000))
    for ci, c in enumerate(classes):
        i = pred_cls == c
        if i.sum() == 0:
            continue
        fpc = (1 - tp[i]).cumsum(0)
        tpc = tp[i].cumsum(0)
        recall = tpc / (n_targets[ci] + eps)
        precision = tpc / (tpc + fpc)
        r_curve[ci] = np.interp(-x, -conf[i], recall[:, 0], left=0)
        p_curve[ci] = np.interp(-x, -conf[i], precision[:, 0], left=1)
        for j in range(tp.shape[1]):
            ap[ci, j] = compute_ap(recall[:, j], precision[:, j])
    f1_curve = 2 * p_curve * r_curve / (p_curve + r_curve + eps)
    best = smooth(f1_curve.mean(0), 0.1).argmax() if len(classes) else 0
    return {"classes": classes, "ap": ap, "p": p_curve[:, best], "r": r_curve[:, best]}


def detection_metrics(pred: pd.DataFrame, gt: pd.DataFrame, file_names: list[str]) -> dict:
    """file_names 사진들의 예측과 정답으로 mAP 네 가지와 클래스별 AP 표를 구한다.

    pred, gt는 yolo_idx와 픽셀 COCO bbox 열을 가진 표다. 반환값의 per_class는 yolo_idx를 인덱스로
    ap50, ap50_95, ap75_95 열을 갖는다(val에 정답이 있는 클래스만).
    """
    gt_by = dict(tuple(gt.groupby("file_name")))
    pred_by = dict(tuple(pred.groupby("file_name")))
    stats = defaultdict(list)
    for name in file_names:
        g, p = gt_by.get(name), pred_by.get(name)
        gt_cls = g["yolo_idx"].to_numpy() if g is not None else np.zeros(0, dtype=int)
        stats["target_cls"].append(gt_cls)
        if p is None or len(p) == 0:
            continue
        p = p.sort_values("conf", ascending=False, kind="stable")
        pred_cls = p["yolo_idx"].to_numpy()
        correct = np.zeros((len(p), len(IOU_THRESHOLDS)), dtype=bool)
        if len(gt_cls):
            iou = box_iou(
                xywh_to_xyxy(g[BBOX_COLS].to_numpy(float)),
                xywh_to_xyxy(p[BBOX_COLS].to_numpy(float)),
            )
            correct = match_predictions(pred_cls, gt_cls, iou)
        stats["tp"].append(correct)
        stats["conf"].append(p["conf"].to_numpy())
        stats["pred_cls"].append(pred_cls)
    target_cls = np.concatenate(stats["target_cls"])
    if not stats["tp"]:
        stats["tp"], stats["conf"], stats["pred_cls"] = [np.zeros((0, 10))], [[]], [[]]
    result = ap_per_class(
        np.concatenate(stats["tp"]).astype(float),
        np.concatenate(stats["conf"]).astype(float),
        np.concatenate(stats["pred_cls"]),
        target_cls,
    )
    ap = result["ap"]
    has_ap = len(ap) > 0
    per_class = pd.DataFrame(
        {"ap50": ap[:, 0], "ap50_95": ap.mean(1), "ap75_95": ap[:, IOU75_COL:].mean(1)},
        index=pd.Index(result["classes"], name="yolo_idx"),
    )
    return {
        MAIN_METRIC: float(per_class["ap75_95"].mean()) if has_ap else 0.0,
        "map50": float(per_class["ap50"].mean()) if has_ap else 0.0,
        "map75": float(ap[:, IOU75_COL].mean()) if has_ap else 0.0,
        "map50_95": float(per_class["ap50_95"].mean()) if has_ap else 0.0,
        "ul_precision": float(result["p"].mean()) if has_ap else 0.0,
        "ul_recall": float(result["r"].mean()) if has_ap else 0.0,
        "per_class": per_class,
    }


def match_boxes(pred: pd.DataFrame, gt: pd.DataFrame, iou_thres: float = POINT_IOU) -> pd.DataFrame:
    """사진마다 신뢰도 높은 예측부터, 짝이 없는 정답 중 IoU가 가장 큰 것과 위치로만 짝짓는다.

    models/evaluate.py의 match_boxes와 같다. 짝마다 예측·정답 category_id, IoU, 정답 행 번호를 낸다.
    """
    rows = []
    gt_by = dict(tuple(gt.groupby("file_name")))
    pred = pred.sort_values("conf", ascending=False)
    for name, p in pred.groupby("file_name", sort=False):
        g = gt_by.get(name)
        if g is None:
            continue
        iou = box_iou(
            xywh_to_xyxy(p[BBOX_COLS].to_numpy(float)), xywh_to_xyxy(g[BBOX_COLS].to_numpy(float))
        )
        free = np.ones(len(g), dtype=bool)
        for i, (pred_idx, pred_category) in enumerate(zip(p.index, p["category_id"], strict=True)):
            j = int(np.where(free, iou[i], -1).argmax())
            if iou[i, j] >= iou_thres and free[j]:
                free[j] = False
                rows.append(
                    [name, pred_idx, g.index[j], pred_category, g["category_id"].iat[j], iou[i, j]]
                )
    cols = ["file_name", "pred_row", "gt_row", "pred_category", "gt_category", "iou"]
    return pd.DataFrame(rows, columns=cols)


def point_metrics(
    pred: pd.DataFrame, gt: pd.DataFrame, conf: float = POINT_CONF, iou_thres: float = POINT_IOU
) -> dict[str, float]:
    """신뢰도 conf 이상 예측의 정밀도·재현율·F1·평균 IoU·다른 약으로 답한 짝 수.

    models/evaluate.py의 point_metrics와 같다.
    """
    pred = pred[pred["conf"] >= conf]
    pairs = match_boxes(pred, gt, iou_thres)
    n_correct = int((pairs["pred_category"] == pairs["gt_category"]).sum())
    precision = n_correct / len(pred) if len(pred) else np.nan
    recall = n_correct / len(gt) if len(gt) else np.nan
    return {
        "precision": precision,
        "recall": recall,
        "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
        "mean_iou": float(pairs["iou"].mean()) if len(pairs) else np.nan,
        "n_wrong_class": len(pairs) - n_correct,
    }


def confusion_matrix(
    pred: pd.DataFrame,
    gt: pd.DataFrame,
    file_names: list[str],
    nc: int,
    conf: float,
    iou_thres: float = CM_IOU,
) -> np.ndarray:
    """(nc+1) × (nc+1) 혼동행렬 [예측, 정답]. 마지막 행·열은 배경이다.

    ultralytics ConfusionMatrix.process_batch와 같은 짝짓기를 쓴다.
    """
    matrix = np.zeros((nc + 1, nc + 1))
    gt_by = dict(tuple(gt.groupby("file_name")))
    pred = pred[pred["conf"] > conf]
    pred_by = dict(tuple(pred.groupby("file_name")))
    for name in file_names:
        g, p = gt_by.get(name), pred_by.get(name)
        gt_cls = g["yolo_idx"].to_numpy() if g is not None else np.zeros(0, dtype=int)
        det_cls = p["yolo_idx"].to_numpy() if p is not None else np.zeros(0, dtype=int)
        if len(gt_cls) == 0 or len(det_cls) == 0:
            for dc in det_cls:
                matrix[dc, nc] += 1
            for gc in gt_cls:
                matrix[nc, gc] += 1
            continue
        iou = box_iou(
            xywh_to_xyxy(g[BBOX_COLS].to_numpy(float)), xywh_to_xyxy(p[BBOX_COLS].to_numpy(float))
        )
        gi, di = np.where(iou > iou_thres)
        matches = np.stack([gi, di, iou[gi, di]], 1)
        if len(matches) > 1:
            matches = matches[matches[:, 2].argsort()[::-1]]
            matches = matches[np.unique(matches[:, 1], return_index=True)[1]]
            matches = matches[matches[:, 2].argsort()[::-1]]
            matches = matches[np.unique(matches[:, 0], return_index=True)[1]]
        gt_match = np.full(len(gt_cls), -1)
        gt_match[matches[:, 0].astype(int)] = matches[:, 1].astype(int)
        for i, gc in enumerate(gt_cls):
            matrix[det_cls[gt_match[i]] if gt_match[i] >= 0 else nc, gc] += 1
        for j in set(range(len(det_cls))) - set(matches[:, 1].astype(int).tolist()):
            matrix[det_cls[j], nc] += 1
    return matrix

## 7. 예측과 fold 평가

`predict_images`는 원본 픽셀 좌표의 예측 표(`PRED_COLS`)와 사진 한 장당 시간(ms)을 돌려준다.
시간은 ultralytics와 같은 세 단계로 나눈다. preprocess = 텐서 변환·장치 이동(파일 읽기 제외),
inference = 모델 forward(모델 안의 resize·정규화, RPN, ROI head, 클래스별 NMS 포함),
postprocess = 결과를 CPU 표로 바꾸기. torchvision은 NMS가 forward 안에 있어 YOLO와 단계 구분이 다르므로
속도 비교는 세 값의 합으로 한다.

In [ ]:
@torch.no_grad()
def predict_images(
    model: torch.nn.Module,
    img_dir: Path,
    file_names: list[str],
    device: str,
    batch_size: int = 4,
    desc: str = "predict",
) -> tuple[pd.DataFrame, dict[str, float]]:
    """사진들을 예측해 (PRED_COLS 표, 사진 한 장당 단계별 ms)를 반환한다.

    모델의 라벨 id에서 1을 빼 yolo_idx로 되돌린다.
    """
    model.eval()
    parts, times = [], dict.fromkeys(SPEED_KEYS, 0.0)
    for start in tqdm(range(0, len(file_names), batch_size), desc=desc, leave=False):
        names = file_names[start : start + batch_size]
        pil_images = []
        for name in names:
            with Image.open(img_dir / name) as img:
                pil_images.append(img.convert("RGB"))
        sync(device)
        t0 = time.perf_counter()
        images = [TF.to_tensor(img).to(device) for img in pil_images]
        sync(device)
        t1 = time.perf_counter()
        outputs = model(images)
        sync(device)
        t2 = time.perf_counter()
        for name, out in zip(names, outputs, strict=True):
            boxes = out["boxes"].cpu().numpy().astype(float)
            idx = out["labels"].cpu().numpy() - 1
            part = pd.DataFrame(
                {
                    "file_name": name,
                    "yolo_idx": idx,
                    "category_id": [IDX_TO_CAT[i] for i in idx],
                    "x": boxes[:, 0],
                    "y": boxes[:, 1],
                    "w": boxes[:, 2] - boxes[:, 0],
                    "h": boxes[:, 3] - boxes[:, 1],
                    "conf": out["scores"].cpu().numpy().astype(float),
                }
            )
            parts.append(part)
        t3 = time.perf_counter()
        for key, dt in zip(SPEED_KEYS, [t1 - t0, t2 - t1, t3 - t2], strict=True):
            times[key] += dt
    pred = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=PRED_COLS)
    speed = {f"ms_{k}": 1000 * v / max(len(file_names), 1) for k, v in times.items()}
    return pred[PRED_COLS], speed


def size_group(n_bboxes: int) -> str:
    """bbox 수로 클래스 그룹 이름을 정한다."""
    return next(name for low, name in SIZE_GROUPS if n_bboxes >= low)


def results_dir(cfg: dict) -> Path:
    """실험의 평가·예측 결과 폴더 <runs_dir>/<exp>/results."""
    return Path(cfg["runs_dir"]) / cfg["exp"] / "results"


def run_stem(cfg: dict, fold: int) -> str:
    """YOLO run 이름 형식 <모델>_<분할>_aug<..>_opt<..>_fold<k>에 스케줄러(lr<..>)를 더한 이름.

    스케줄러를 켜고 끈 run이 한 실험 폴더에서 섞이지 않고 summarize_exp에서 따로 묶이게 한다.
    """
    aug = "off" if cfg["hflip_p"] == 0 else f"flip{cfg['hflip_p']:g}"
    lr = "cos" if cfg["cos_lr"] else "linear"
    external = ""
    if external_enabled(cfg):
        options = cfg["external_data"]
        external = f"_aihub{options['target_bbox']}{'cls' if options['extra_as_class'] else 'bg'}"
    return (
        f"frcnn_r50fpnv2_{split_name(cfg)}_aug{aug}_opt{cfg['optimizer']}_lr{lr}"
        f"{external}_fold{fold}"
    )


def fold_files(fold: int, split: str) -> list[str]:
    """fold의 train 또는 val 사진 이름 목록."""
    rows = FOLDS[(FOLDS["fold"] == fold) & (FOLDS["split"] == split)]
    return sorted(rows["file_name"])


def evaluate_fold(
    model: torch.nn.Module, cfg: dict, fold: int, weights: Path, device: str = DEVICE
) -> tuple[pd.DataFrame, dict]:
    """fold의 val로 평가해 (클래스별 표, 요약)을 반환하고 results 폴더에 저장한다.

    요약 열은 models/evaluate.py의 evaluate_fold와 같다. val 예측은 <stem>_val_pred.csv로도 남긴다.
    """
    stem = run_stem(cfg, fold)
    val_files = fold_files(fold, "val")
    val_gt = LABELS[LABELS["file_name"].isin(val_files)]
    pred, speed = predict_images(
        model, TRAIN_IMG_DIR, val_files, device, cfg["eval_batch"], desc=f"val fold{fold}"
    )
    # 외부 전용 클래스 예측은 평가에서 뺀다. 저장하는 val 예측은 15절 OCR이 쓰도록 그대로 둔다.
    pred_eval = competition_pred(pred)
    metric = detection_metrics(pred_eval, val_gt, val_files)

    per_class = CLASS_MAP.copy()
    per_class["group"] = per_class["category_id"].map(
        LABELS["category_id"].value_counts().map(size_group)
    )
    split_of = LABELS["file_name"].map(FOLDS[FOLDS["fold"] == fold].set_index("file_name")["split"])
    for split in ["train", "val"]:
        counts = (
            training_counts(cfg, fold)
            if split == "train"
            else LABELS.loc[split_of == split, "category_id"].value_counts()
        )
        per_class[f"n_{split}_bbox"] = per_class["category_id"].map(counts).fillna(0).astype(int)
    ap_cols = ["ap50", "ap50_95", "ap75_95"]
    per_class = per_class.join(metric["per_class"], on="yolo_idx")
    per_class.loc[per_class["n_val_bbox"] == 0, ap_cols] = np.nan

    # 닮은 약 쌍 혼동: YOLO 평가처럼 평가 conf(0.001)로 만든 혼동행렬에서 센다
    matrix = confusion_matrix(pred_eval, val_gt, val_files, len(CLASS_MAP), cfg["conf"])
    pairs = {
        f"confuse_{a}_{b}": int(
            matrix[CAT_TO_IDX[a], CAT_TO_IDX[b]] + matrix[CAT_TO_IDX[b], CAT_TO_IDX[a]]
        )
        for a, b in CONFUSION_PAIRS
    }
    summary = {
        "run": stem.rsplit("_fold", 1)[0],
        "weights": str(weights),
        "label_version": cfg["label_version"],
        "split_name": split_name(cfg),
        "fold": fold,
        **{k: v for k, v in metric.items() if k in {MAIN_METRIC, "map50", "map75", "map50_95"}},
        **speed,
        **point_metrics(pred_eval, val_gt),
        **{
            f"ap75_95_{name}": float(per_class.loc[per_class["group"] == name, "ap75_95"].mean())
            for _, name in SIZE_GROUPS
        },
        "n_classes_not_in_val": int((per_class["n_val_bbox"] == 0).sum()),
        "n_val_only_classes": int(
            ((per_class["n_val_bbox"] > 0) & (per_class["n_train_bbox"] == 0)).sum()
        ),
        **pairs,
        "n_extra_class_pred": len(pred) - len(pred_eval),
    }
    out_dir = results_dir(cfg)
    out_dir.mkdir(parents=True, exist_ok=True)
    per_class.to_csv(out_dir / f"{stem}_per_class.csv", index=False, encoding="utf-8")
    pd.DataFrame([summary]).to_csv(out_dir / f"{stem}_summary.csv", index=False, encoding="utf-8")
    pred.to_csv(out_dir / f"{stem}_val_pred.csv", index=False, encoding="utf-8")
    return per_class, summary

## 8. 학습 루프

- epoch마다 train → val 평가(mAP75-95)를 반복한다. 학습률은 iteration마다 바꾼다.
  처음 3 epoch는 warmup하고, 이후 `cos_lr`로 고른다(True: 코사인 감소, False: 선형 감소).
  켜고 끈 결과는 같은 `exp`로 돌려도 run 이름(`_lrcos` / `_lrlinear`)으로 나뉘어 한 표에서 비교된다.
- val mAP75-95가 가장 높은 epoch의 가중치를 `weights/best.pt`, 매 epoch 가중치를 `weights/last.pt`로 저장한다.
- val mAP75-95가 `patience` epoch 동안 좋아지지 않으면 멈춘다.
- epoch마다 val 손실도 잰다. torchvision 검출 모델은 train 모드에서만 손실을 내므로 train 모드로
  두되 BatchNorm은 eval로 바꿔 val 사진이 running 통계를 바꾸지 않게 한다.
- epoch별 손실(`train/*`, `val/*`)·지표는 `results.csv`, 실행 설정은 `config.json`에 남긴다.
- `plot_curves(cfg)`는 그 `results.csv`로 fold별 학습 곡선을 그린다. 학습이 끝난 fold만 그리므로
  9절 동작 확인이나 fold 하나만 돌린 뒤에도 볼 수 있다.
- 학습이 끝나면 `best.pt`로 `evaluate_fold`를 돌린다. 이미 요약이 있는 fold는 건너뛴다(`skip_done`).

In [ ]:
def make_scheduler(
    optimizer: torch.optim.Optimizer, cfg: dict, iters_per_epoch: int
) -> torch.optim.lr_scheduler.LambdaLR:
    """warmup(선형) 뒤 lr을 정하는 iteration 단위 스케줄러.

    워밍업 이후 lr0에서 lr0 × lrf까지 줄인다(cos_lr=True는 코사인, False는 선형).
    """
    total = cfg["epochs"] * iters_per_epoch
    warmup = min(max(1, round(cfg["warmup_epochs"] * iters_per_epoch)), total)

    def factor(it: int) -> float:
        if it < warmup:
            return cfg["warmup_factor"] + (1 - cfg["warmup_factor"]) * it / warmup
        progress = (it - warmup) / max(1, total - warmup)
        if not cfg["cos_lr"]:
            return 1 - (1 - cfg["lrf"]) * progress
        return cfg["lrf"] + (1 - cfg["lrf"]) * 0.5 * (1 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


def train_one_epoch(
    model: torch.nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LambdaLR,
    device: str,
    scaler: torch.amp.GradScaler | None,
    desc: str,
) -> dict[str, float]:
    """한 epoch 학습하고 손실 평균을 반환한다. 손실이 NaN·inf가 되면 멈춘다."""
    model.train()
    sums, n = defaultdict(float), 0
    for images, targets, _ in tqdm(loader, desc=desc, leave=False):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        amp = torch.autocast("cuda") if scaler is not None else contextlib.nullcontext()
        with amp:
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
        if not torch.isfinite(loss):
            raise FloatingPointError(f"손실이 {loss.item()}입니다: {loss_dict}")
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        scheduler.step()
        for key, value in loss_dict.items():
            sums[key] += value.item()
        sums["loss"] += loss.item()
        n += 1
    return {key: value / max(n, 1) for key, value in sums.items()}


@torch.no_grad()
def val_losses(
    model: torch.nn.Module, loader: DataLoader, device: str, amp: bool, desc: str
) -> dict[str, float]:
    """val 손실 평균을 반환한다(키는 train_one_epoch와 같다).

    손실은 train 모드에서만 나오므로 train 모드로 두되, BatchNorm은 eval로 바꿔
    val 사진이 running 통계를 바꾸지 않게 한다.
    """
    model.train()
    for module in model.modules():
        if isinstance(module, torch.nn.BatchNorm2d):
            module.eval()
    sums, n = defaultdict(float), 0
    for images, targets, _ in tqdm(loader, desc=desc, leave=False):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        with torch.autocast("cuda") if amp else contextlib.nullcontext():
            loss_dict = model(images, targets)
        for key, value in loss_dict.items():
            sums[key] += value.item()
        sums["loss"] += sum(loss_dict.values()).item()
        n += 1
    return {key: value / max(n, 1) for key, value in sums.items()}


def run_config(cfg: dict, fold: int, n_train: int, n_val: int) -> dict:
    """결과와 함께 남길 실행 설정 전체(장치·버전·데이터 크기 포함)."""
    return {
        **cfg,
        "fold": fold,
        "split_name": split_name(cfg),
        "num_classes": model_num_classes(cfg),
        "n_train_images": n_train,
        "n_val_images": n_val,
        "device": DEVICE,
        "python": platform.python_version(),
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "created_at": datetime.now().astimezone().isoformat(timespec="seconds"),
    }


def train_fold(cfg: dict, fold: int, device: str = DEVICE) -> Path:
    """fold 하나를 학습해 best.pt 경로를 반환한다. 설정·epoch 기록은 run 폴더에 남긴다."""
    seed_everything(cfg["seed"])
    stem = run_stem(cfg, fold)
    run_dir = Path(cfg["runs_dir"]) / cfg["exp"] / stem
    weights_dir = run_dir / "weights"
    weights_dir.mkdir(parents=True, exist_ok=True)

    train_files, external_files, external_bundle = training_data_spec(cfg, fold)
    val_files = fold_files(fold, "val")
    val_gt = LABELS[LABELS["file_name"].isin(val_files)]
    training_dataset = PillDataset(TRAIN_IMG_DIR, train_files, LABELS, hflip_p=cfg["hflip_p"])
    if external_bundle is not None:
        training_dataset = ConcatDataset(
            [
                training_dataset,
                ExternalPillDataset(external_files, external_bundle, hflip_p=cfg["hflip_p"]),
            ]
        )
    config = run_config(cfg, fold, len(training_dataset), len(val_files))
    if external_bundle is not None:
        config.update(
            {
                "n_original_train_images": len(train_files),
                "n_external_train_images": len(external_files),
                "n_external_train_bbox": int(
                    external_bundle["labels"]["file_name"].isin(external_files).sum()
                ),
                "external_fingerprint": external_bundle["fingerprint"],
                "external_folders": external_bundle["folders"],
                "external_only_classes": (
                    "class" if cfg["external_data"]["extra_as_class"] else "background"
                ),
                "extra_class_map": external_bundle["extra_map"].to_dict("records"),
            }
        )
    (run_dir / "config.json").write_text(
        json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(
        f"[{stem}] train {len(train_files)}장 + 외부 {len(external_files)}장, "
        f"val {len(val_files)}장, device {device}"
    )

    loader = DataLoader(
        training_dataset,
        batch_size=cfg["batch"],
        shuffle=True,
        num_workers=cfg["workers"],
        collate_fn=collate,
        generator=torch.Generator().manual_seed(cfg["seed"]),
    )
    val_loader = DataLoader(
        PillDataset(TRAIN_IMG_DIR, val_files, LABELS),
        batch_size=cfg["eval_batch"],
        num_workers=cfg["workers"],
        collate_fn=collate,
    )
    model = build_model(cfg).to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    if cfg["optimizer"] != "SGD":
        raise ValueError(f"지원하지 않는 옵티마이저: {cfg['optimizer']}")
    optimizer = torch.optim.SGD(
        params, lr=cfg["lr0"], momentum=cfg["momentum"], weight_decay=cfg["weight_decay"]
    )
    scheduler = make_scheduler(optimizer, cfg, len(loader))
    scaler = torch.amp.GradScaler("cuda") if cfg["amp"] and device == "cuda" else None

    history, best, stale = [], -1.0, 0
    for epoch in range(1, cfg["epochs"] + 1):
        start = time.perf_counter()
        losses = train_one_epoch(
            model, loader, optimizer, scheduler, device, scaler, f"fold{fold} epoch {epoch}"
        )
        val_loss = val_losses(model, val_loader, device, scaler is not None, f"fold{fold} val loss")
        pred, _ = predict_images(model, TRAIN_IMG_DIR, val_files, device, cfg["eval_batch"])
        metric = detection_metrics(competition_pred(pred), val_gt, val_files)
        row = {
            "epoch": epoch,
            **{f"train/{k}": v for k, v in losses.items()},
            **{f"val/{k}": v for k, v in val_loss.items()},
            "lr": optimizer.param_groups[0]["lr"],
            **{k: v for k, v in metric.items() if k != "per_class"},
            "time_s": time.perf_counter() - start,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / "results.csv", index=False, encoding="utf-8")
        ckpt = {"model": model.state_dict(), "epoch": epoch, "metrics": row, "config": config}
        torch.save(ckpt, weights_dir / "last.pt")
        improved = metric[MAIN_METRIC] > best
        if improved:
            best, stale = metric[MAIN_METRIC], 0
            torch.save(ckpt, weights_dir / "best.pt")
        else:
            stale += 1
        print(
            f"epoch {epoch:3d}  loss {losses['loss']:.4f}  val_loss {val_loss['loss']:.4f}  "
            f"mAP75-95 {metric[MAIN_METRIC]:.4f}  "
            f"mAP50 {metric['map50']:.4f}  {row['time_s']:.0f}s{'  *best' if improved else ''}"
        )
        if stale >= cfg["patience"]:
            print(f"early stopping: {cfg['patience']} epoch 동안 mAP75-95가 좋아지지 않음")
            break
    return weights_dir / "best.pt"


def load_model(weights: Path, cfg: dict, device: str = DEVICE) -> torch.nn.Module:
    """best.pt를 읽어 평가 모드 모델을 만든다. 사전학습 가중치는 받지 않는다."""
    ckpt = torch.load(weights, map_location="cpu", weights_only=False)
    model = build_model({**cfg, "pretrained": None}, ckpt["config"]["num_classes"])
    model.load_state_dict(ckpt["model"])
    return model.to(device).eval()


def run_folds(cfg: dict, skip_done: bool = True) -> pd.DataFrame:
    """cfg["folds"]를 차례로 학습·평가하고 fold 요약 표를 반환한다."""
    summaries = []
    for fold in cfg["folds"]:
        stem = run_stem(cfg, fold)
        summary_path = results_dir(cfg) / f"{stem}_summary.csv"
        if skip_done and summary_path.is_file():
            if external_enabled(cfg):
                config_path = results_dir(cfg) / f"{stem}_config.json"
                saved = json.loads(config_path.read_text(encoding="utf-8"))
                if saved.get("external_fingerprint") != get_external_bundle(cfg)["fingerprint"]:
                    raise ValueError(
                        "외부 데이터가 이전 실행과 다릅니다. 새 exp 이름을 지정하세요."
                    )
            print(f"[{stem}] 이미 평가 결과가 있어 건너뜀")
            summaries.append(pd.read_csv(summary_path).iloc[0].to_dict())
            continue
        weights = train_fold(cfg, fold)
        model = load_model(weights, cfg)
        _, summary = evaluate_fold(model, cfg, fold, weights)
        (results_dir(cfg) / f"{stem}_config.json").write_text(
            (weights.parent.parent / "config.json").read_text(encoding="utf-8"), encoding="utf-8"
        )
        summaries.append(summary)
        del model
    return pd.DataFrame(summaries)

In [ ]:
LOSS_COLORS = {"train": "#2a78d6", "val": "#eb6834"}
CURVE_METRICS = {"map50": "mAP50", "map50_95": "mAP50-95", MAIN_METRIC: "mAP75-95"}
CURVE_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]


def plot_curves(cfg: dict) -> None:
    """fold별 results.csv로 train·val 손실(위)과 val mAP(아래)를 epoch별 꺾은선으로 그린다.

    val 손실을 남기기 전에 학습한 run은 train 손실만 그린다. 점선은 mAP75-95가 가장 높은
    epoch(best.pt)이다. 그림은 results 폴더에 <run>_curves.png로 저장한다.
    """
    histories = {}
    for fold in cfg["folds"]:
        path = Path(cfg["runs_dir"]) / cfg["exp"] / run_stem(cfg, fold) / "results.csv"
        if path.is_file():
            histories[fold] = pd.read_csv(path)
    if not histories:
        raise FileNotFoundError(f"{cfg['exp']}에 results.csv가 있는 fold가 없습니다.")

    fig, axes = plt.subplots(
        2, len(histories), figsize=(max(4 * len(histories), 8), 7), sharey="row", squeeze=False
    )
    for col, (fold, hist) in enumerate(histories.items()):
        best = int(hist.loc[hist[MAIN_METRIC].idxmax(), "epoch"])
        style = {"lw": 2, "marker": "o" if len(hist) < 10 else None}  # epoch가 적으면 점도 찍음
        loss_ax, map_ax = axes[0, col], axes[1, col]
        for split, color in LOSS_COLORS.items():
            if f"{split}/loss" in hist:
                loss_ax.plot(
                    hist["epoch"], hist[f"{split}/loss"], color=color, label=split, **style
                )
        for (key, name), color in zip(CURVE_METRICS.items(), CURVE_COLORS, strict=True):
            map_ax.plot(hist["epoch"], hist[key], color=color, label=name, **style)
        for ax in (loss_ax, map_ax):
            ax.axvline(best, color="gray", ls="--", lw=1)
            ax.grid(alpha=0.3)
            ax.spines[["top", "right"]].set_visible(False)
            ax.set_xlim(0.5, hist["epoch"].max() + 0.5)
            ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True, min_n_ticks=1))
        loss_ax.set_title(f"fold {fold} (best epoch {best}, {hist[MAIN_METRIC].max():.3f})")
        map_ax.set_xlabel("epoch")
        map_ax.set_ylim(0, 1)
    axes[0, 0].set_ylabel("loss")
    axes[1, 0].set_ylabel("val mAP")
    axes[0, 0].legend()
    axes[1, 0].legend()
    run = run_stem(cfg, cfg["folds"][0]).rsplit("_fold", 1)[0]
    fig.suptitle(f"{run} 학습 곡선")
    fig.tight_layout()
    out_dir = results_dir(cfg)
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / f"{run}_curves.png", dpi=120)
    plt.show()

## 9. 동작 확인 (fold 0, 1 epoch, train 10%)

학습 → 평가 → 결과 저장 → 학습 곡선이 끝까지 도는지만 본다. 점수는 의미 없다.
1 epoch이면 곡선이 점 하나로 나온다. 곡선 모양을 보려면 `epochs`를 늘린다.

In [ ]:
SMOKE_CFG = {
    **CFG,
    "exp": "frcnn_aihub_ocr_smoke",
    "folds": [0],
    "epochs": 1,
    "fraction": 0.1,
    "cos_lr": False,
    "ocr": {**CFG["ocr"], "epochs": 1},
}
smoke = run_folds(SMOKE_CFG, skip_done=False)
display(smoke.T)
plot_curves(SMOKE_CFG)

## 10. 본 학습

시간이 오래 걸리므로 `RUN_FULL = True`로 바꿔야 돈다. 끊겼다가 다시 돌리면 평가가 끝난 fold는 건너뛴다.

In [ ]:
RUN_FULL = True
if RUN_FULL:
    full = run_folds(
        {
            **CFG,
            "folds": [0],
            "cos_lr": False,
        }
    )
    display(full.T)

In [ ]:
RUN_FULL = False
if RUN_FULL:
    full = run_folds(CFG)
    display(full.T)

## 11. 5 fold 요약

`models/evaluate.py`의 `summarize_exp`와 같은 방식으로 run별 평균·표준편차를 `summary.csv`에 저장한다.
같은 `results/` 폴더에 YOLO 결과가 있으면 한 표에 함께 나온다.

In [ ]:
def summarize_exp(cfg: dict) -> pd.DataFrame:
    """results 폴더의 fold 요약을 run별 평균·표준편차로 모아 summary.csv에 저장한다."""
    out_dir = results_dir(cfg)
    runs = pd.concat(
        [pd.read_csv(p) for p in sorted(out_dir.glob("*_summary.csv"))], ignore_index=True
    )
    numeric = runs.drop(columns=["fold"]).select_dtypes("number").columns
    summary = runs.groupby("run")[list(numeric)].agg(["mean", "std"])
    summary.insert(0, ("n_folds", "count"), runs.groupby("run")["fold"].nunique())
    summary = summary.sort_values((MAIN_METRIC, "mean"), ascending=False)
    summary.to_csv(out_dir / "summary.csv", encoding="utf-8")
    return summary


REPORT_CFG = CFG if RUN_FULL else SMOKE_CFG
summarize_exp(REPORT_CFG)[[MAIN_METRIC, "map50", "map50_95", "precision", "recall"]]

## 12. 학습 곡선

fold별로 epoch마다 train·val 손실(위)과 val mAP50·mAP50-95·mAP75-95(아래)를 그린다.
점선은 mAP75-95가 가장 높은 epoch(`best.pt`)이다. 각 run의 `results.csv`를 읽으므로
학습이 끝난 뒤 커널을 다시 켜도 그릴 수 있다. 함수는 8절에 있다. val 손실을 기록하기 전에 학습한 run은 train 손실만 나온다.

In [ ]:
plot_curves(REPORT_CFG)

## 13. 혼동행렬

conf 0.001과 0.25로 그려 비교한다. 보기 좋은 쪽을 정하면 `CM_CONF`에 고정한다.
행 = 예측, 열 = 정답, 마지막 행·열은 배경(놓침·오검출). 축 이름은 category_id다.

In [ ]:
REPORT_FOLD = REPORT_CFG["folds"][0]


def load_val_pred(cfg: dict, fold: int) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """evaluate_fold가 저장한 val 예측(대회 56종만)과 그 fold의 val 정답·사진 목록을 읽는다."""
    pred = competition_pred(pd.read_csv(results_dir(cfg) / f"{run_stem(cfg, fold)}_val_pred.csv"))
    val_files = fold_files(fold, "val")
    return pred, LABELS[LABELS["file_name"].isin(val_files)], val_files


def plot_confusion(matrix: np.ndarray, title: str, ax: plt.Axes) -> None:
    """혼동행렬을 log 색으로 그린다. 대각선이 진할수록 좋다."""
    names = [*map(str, CLASS_MAP["category_id"]), "bg"]
    ax.imshow(np.log1p(matrix), cmap="Blues")
    ax.set_xticks(range(len(names)), names, rotation=90, fontsize=5)
    ax.set_yticks(range(len(names)), names, fontsize=5)
    ax.set_xlabel("정답")
    ax.set_ylabel("예측")
    ax.set_title(title)


val_pred, val_gt, val_files = load_val_pred(REPORT_CFG, REPORT_FOLD)
plot_dir = results_dir(REPORT_CFG) / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
for ax, cm_conf in zip(axes, [0.001, 0.25], strict=True):
    matrix = confusion_matrix(val_pred, val_gt, val_files, len(CLASS_MAP), cm_conf)
    plot_confusion(matrix, f"{run_stem(REPORT_CFG, REPORT_FOLD)} · conf {cm_conf:g}", ax)
fig.tight_layout()
fig.savefig(plot_dir / f"{run_stem(REPORT_CFG, REPORT_FOLD)}_confusion.png", dpi=120)
plt.show()

## 14. 실패 사례

고정 기준(신뢰도 0.5 이상, IoU 0.5 이상 짝)으로 val 예측을 정답과 짝짓고 세 가지 실패를 그린다.

- 놓침: 짝이 없는 정답 (초록 점선)
- 다른 약: 짝은 지었지만 약이 틀림 (빨강)
- 위치 어긋남: 약은 맞지만 IoU < 0.75 (주황). 주 지표 mAP75-95에서 점수를 받지 못한다.

정답은 초록 실선, 짝이 없는 예측(오검출)은 보라 점선이다. 실패가 많은 사진부터 그린다.

In [ ]:
LOOSE_IOU = 0.75


def failure_table(pred: pd.DataFrame, gt: pd.DataFrame) -> pd.DataFrame:
    """정답마다 실패 종류(missed, wrong_class, loose_box, ok)와 짝지은 예측을 붙인 표."""
    pred = pred[pred["conf"] >= POINT_CONF]
    pairs = match_boxes(pred, gt).set_index("gt_row")
    table = gt.copy()
    table["pred_row"] = table.index.map(pairs["pred_row"])
    table["pred_category"] = table.index.map(pairs["pred_category"])
    table["iou"] = table.index.map(pairs["iou"])
    table["failure"] = np.select(
        [
            table["pred_row"].isna(),
            table["pred_category"] != table["category_id"],
            table["iou"] < LOOSE_IOU,
        ],
        ["missed", "wrong_class", "loose_box"],
        "ok",
    )
    return table


def plot_failures(
    pred: pd.DataFrame, gt: pd.DataFrame, n_images: int = 6, ncols: int = 3
) -> plt.Figure | None:
    """실패가 많은 사진 n_images장에 정답·예측 박스를 그린다."""
    table = failure_table(pred, gt)
    shown = pred[pred["conf"] >= POINT_CONF]
    unmatched = shown.drop(index=table["pred_row"].dropna().astype(int))
    n_fail = (table["failure"] != "ok").groupby(table["file_name"]).sum()
    n_fail = n_fail.add(unmatched["file_name"].value_counts(), fill_value=0)
    files = n_fail[n_fail > 0].sort_values(ascending=False).index[:n_images]
    if len(files) == 0:
        print("실패 사례가 없습니다.")
        return None
    print(table["failure"].value_counts().to_dict(), f"오검출 {len(unmatched)}개")
    nrows = math.ceil(len(files) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows), squeeze=False)
    colors = {"wrong_class": "red", "loose_box": "orange"}
    for ax, name in zip(axes.flat, files, strict=False):
        with Image.open(TRAIN_IMG_DIR / name) as img:
            ax.imshow(img.convert("RGB"))
        for _, row in table[table["file_name"] == name].iterrows():
            style = "--" if row["failure"] == "missed" else "-"
            ax.add_patch(
                Rectangle((row.x, row.y), row.w, row.h, fill=False, ec="lime", lw=2, ls=style)
            )
            ax.text(row.x, row.y - 5, str(row.category_id), color="lime", fontsize=8)
            if row["failure"] in colors:
                p = shown.loc[int(row["pred_row"])]
                color = colors[row["failure"]]
                ax.add_patch(Rectangle((p.x, p.y), p.w, p.h, fill=False, ec=color, lw=2))
                ax.text(
                    p.x,
                    p.y + p.h + 25,
                    f"{p.category_id} {p.conf:.2f} IoU {row.iou:.2f}",
                    color=color,
                    fontsize=8,
                )
        for _, p in unmatched[unmatched["file_name"] == name].iterrows():
            ax.add_patch(Rectangle((p.x, p.y), p.w, p.h, fill=False, ec="violet", lw=2, ls=":"))
            ax.text(
                p.x, p.y + p.h + 25, f"{p.category_id} {p.conf:.2f}", color="violet", fontsize=8
            )
        ax.set_title(name, fontsize=8)
        ax.axis("off")
    for ax in list(axes.flat)[len(files) :]:
        ax.axis("off")
    fig.tight_layout()
    return fig


fig = plot_failures(val_pred, val_gt)
if fig is not None:
    fig.savefig(plot_dir / f"{run_stem(REPORT_CFG, REPORT_FOLD)}_failures.png", dpi=100)
    plt.show()

## 15. 각인 OCR 학습과 클래스 재분류

알약에 새겨진 식별문자(각인)를 읽는 작은 OCR 모델을 **탐지기와 같은 학습 사진으로 따로 학습**하고, 탐지 bbox의
클래스를 OCR로 다시 고른다. 방법은 `exp/yolo26m_nms_free_aihub_copypaste_eda_ocr` 노트북 5-2절과 같다.
탐지기 가중치와 학습은 위 그대로다. `CFG["ocr"]["enabled"] = False`면 건너뛴다.

1. **정답 텍스트**: bbox마다 원본 JSON의 `drug_dir`(이 사진에서 보이는 면)에 따라 `print_front` 또는
   `print_back`을 쓴다. 설명어는 한 글자로 바꾼다(`십자분할선`→`+`, `분할선`→`|`, `마크`→`@`).
   공백은 지우고 영문은 대문자로 맞춘다. 예: `I분할선Y` → `I|Y`, `마크 PGN    150` → `@PGN150`.
   면을 모르는 bbox(대회 보정에서 추가한 bbox)는 OCR 학습에서 뺀다.
2. **학습 데이터**: 탐지기와 같은 fold의 학습 사진(대회 train + AI Hub, `fraction` 적용)의 정답 bbox crop이다.
   val 사진은 쓰지 않는다. bbox 중심에서 긴 변의 1.3배 정사각형을 잘라, 학습 때마다 임의 각도로 돌리고
   bbox 크기·위치를 ±10%·±5% 흔들고 밝기·대비를 바꾼다. 알약이 아무 방향으로 놓이므로 회전을 학습한다.
3. **모델**: CRNN(VGG형 CNN + 2층 BiLSTM)을 처음부터 CTC loss로 학습한다(AdamW, one-cycle 코사인 학습률).
   best는 fold val 정답 crop의 **각인 분류 정확도**(아래 4로 고른 클래스가 정답과 같은 비율)가 가장 높은 epoch다.
4. **클래스 점수**: crop을 `rotations` 방향으로 돌려 보고, 모델 클래스(외부 전용 포함)마다 앞·뒷면 각인이
   나올 CTC 로그 확률 중 가장 큰 값을 그 클래스 점수로 쓴다. 이것을 `temperature`로 나눠 softmax한 것이 OCR 확률이다.
   각인이 없는 면은 빈 문자열이라 "글자가 안 보임"도 하나의 후보가 된다. 각인이 같은 약끼리는 확률이 나뉜다.
5. **재분류**: 탐지 bbox마다 OCR이 가리키는 클래스 확률이 `switch_prob` 이상이고, 탐지기가 고른 클래스의
   OCR 확률이 `keep_prob` 이하일 때만 클래스를 바꾼다. 박스 좌표와 점수는 탐지기 값을 그대로 둔다.
   외부 전용 클래스로 바뀐 예측은 평가에서 뺀다(7절과 같은 규칙).

평가는 7절 `evaluate_fold`가 저장한 val 예측(conf 0.001, 외부 전용 클래스 포함)을 그대로 쓰고, 6절과 같은 계산으로
OCR 전·후 mAP75-95를 비교한다. OCR 전 값이 7절 요약 값과 같은지 함께 출력한다.
대상 run은 10절 본 학습(fold 0, 선형 감소)이고, 그 결과가 없으면 9절 동작 확인 run을 쓴다.
결과는 `runs/<exp>/results/ocr/<run>/`(가중치·epoch 기록·그래프·각인 사전·val 재분류 표)에 남는다.

In [ ]:
import gc
import unicodedata
from multiprocessing.pool import ThreadPool

import cv2

# 각인 설명어 → 한 글자 토큰. 긴 말부터 바꾼다(십자분할선 안의 분할선).
OCR_TOKENS = {"십자분할선": "+", "분할선": "|", "마크": "@"}
FACE_COLUMNS = {"앞면": "print_front", "뒷면": "print_back"}
OCR_TEXT_COLUMNS = ["drug_dir", *FACE_COLUMNS.values()]
OCR_CROP_MARGIN = 1.3  # crop은 bbox 긴 변의 이 배수 정사각형. 회전·흔들기에 쓸 여백이다
OCR_SCORE_BATCH = 64  # 클래스 점수를 계산할 때 한 번에 넣는 crop 수(회전 수만큼 늘어남)
OCR_CROP_CHUNK = 2048  # 예측 bbox crop을 이 개수씩 읽는다. 메모리 상한이고 결과와는 무관하다
OCR_THREADS = min(8, os.cpu_count() or 1)
# OCR 학습 결과에 영향을 주는 CFG["ocr"] 값. 저장된 OCR 기록과 다르면 다시 학습한다.
OCR_TRAINING_KEYS = ["imgsz", "epochs", "batch", "lr", "patience"]


def empty_device_cache() -> None:
    """GPU 메모리 캐시를 비운다."""
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()


def ocr_dir(cfg: dict, fold: int) -> Path:
    """run의 OCR 결과 폴더 <runs_dir>/<exp>/results/ocr/<run>."""
    return results_dir(cfg) / "ocr" / run_stem(cfg, fold)


def normalize_imprint(text: object) -> str:
    """각인 문자열의 설명어를 한 글자 토큰으로 바꾸고 공백을 지운 뒤 대문자로 맞춘다."""
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    for word, token in OCR_TOKENS.items():
        text = text.replace(word, token)
    return "".join(text.split()).upper()


def visible_texts(boxes: pd.DataFrame) -> pd.Series:
    """bbox마다 보이는 면(drug_dir)의 정규화 각인. 면을 모르면 NaN."""
    texts = pd.Series(np.nan, index=boxes.index, dtype=object)
    for face, column in FACE_COLUMNS.items():
        mask = boxes["drug_dir"].eq(face)
        texts[mask] = boxes.loc[mask, column].map(normalize_imprint)
    return texts


def ocr_box_table(cfg: dict, fold: int) -> pd.DataFrame:
    """대회 v1 라벨과 외부 학습 라벨의 bbox마다 사진 경로·나눔·보이는 면 각인을 붙인다.

    split은 탐지기와 같은 학습 사진이면 train, fold의 val이면 val, fraction으로 빠진 사진은 unused.
    """
    base_names, external_names, bundle = training_data_spec(cfg, fold)
    split_of = {name: "train" for name in base_names}
    split_of.update({name: "val" for name in fold_files(fold, "val")})
    tables = [
        LABELS.assign(
            path=[TRAIN_IMG_DIR / name for name in LABELS["file_name"]],
            split=LABELS["file_name"].map(split_of).fillna("unused"),
            source="competition",
        )
    ]
    if bundle is not None:
        external = bundle["labels"].copy()
        external["path"] = external["file_name"].map(bundle["paths"])
        used = external["file_name"].isin(set(external_names))
        external["split"] = np.where(used, "train", "unused")
        external["source"] = "aihub"
        tables.append(external)
    columns = ["file_name", "path", "split", "source", "category_id", *BBOX_COLS]
    columns += OCR_TEXT_COLUMNS
    boxes = pd.concat([table.reindex(columns=columns) for table in tables], ignore_index=True)
    boxes["category_id"] = boxes["category_id"].astype(int)
    boxes["text"] = visible_texts(boxes)
    return boxes


def imprint_dictionary(boxes: pd.DataFrame, class_ids: list[int]) -> pd.DataFrame:
    """모델 클래스마다 앞·뒷면 정규화 각인 표. 약의 각인이 JSON마다 다르면 가장 많은 쪽을 쓴다."""
    known = boxes.dropna(subset=["drug_dir"]).copy()
    for column in FACE_COLUMNS.values():
        known[column] = known[column].map(normalize_imprint)
    counts = (
        known.groupby(["category_id", "print_front", "print_back"]).size().rename("n").reset_index()
    )
    counts = counts.sort_values(["category_id", "n"], ascending=[True, False])
    conflicts = sorted(
        int(value) for value in counts.loc[counts["category_id"].duplicated(), "category_id"]
    )
    if conflicts:
        print(f"JSON마다 각인이 다른 클래스 {len(conflicts)}종(가장 많은 각인 사용): {conflicts}")
    table = counts.drop_duplicates("category_id").set_index("category_id")
    if missing := sorted(set(class_ids) - set(table.index)):
        raise ValueError(f"각인 정보가 없는 모델 클래스가 있습니다: {missing}")
    return table.loc[class_ids, ["print_front", "print_back"]]


def ctc_length(text: str) -> int:
    """CTC가 text를 내는 데 필요한 최소 출력 길이(같은 글자가 이어지면 사이에 blank가 필요)."""
    return len(text) + sum(left == right for left, right in zip(text, text[1:], strict=False))


def square_crop(image: np.ndarray, box: np.ndarray, size: int) -> np.ndarray:
    """bbox(x, y, w, h) 중심에서 긴 변 × OCR_CROP_MARGIN 정사각형을 size×size로 자른다."""
    x, y, w, h = box
    scale = size / (max(w, h) * OCR_CROP_MARGIN)
    matrix = np.array(
        [
            [scale, 0, size / 2 - (x + w / 2) * scale],
            [0, scale, size / 2 - (y + h / 2) * scale],
        ]
    )
    return cv2.warpAffine(image, matrix, (size, size), borderMode=cv2.BORDER_REPLICATE)


def load_crops(boxes: pd.DataFrame, imgsz: int) -> np.ndarray:
    """bbox마다 square_crop을 읽어 boxes 순서대로 (N, S, S, 3) uint8 BGR로 반환한다."""
    size = round(imgsz * OCR_CROP_MARGIN)

    def read(item: tuple[str, pd.DataFrame]) -> list[tuple[int, np.ndarray]]:
        path, rows = item
        image = cv2.imread(str(path))
        if image is None:
            raise FileNotFoundError(f"이미지를 읽을 수 없습니다: {path}")
        return [
            (index, square_crop(image, box, size))
            for index, box in zip(rows.index, rows[BBOX_COLS].to_numpy(float), strict=True)
        ]

    boxes = boxes.reset_index(drop=True)
    groups = list(boxes.groupby(boxes["path"].astype(str), sort=False))
    with ThreadPool(OCR_THREADS) as pool:
        results = [pair for pairs in pool.map(read, groups) for pair in pairs]
    crops = np.empty((len(boxes), size, size, 3), dtype=np.uint8)
    for index, crop in results:
        crops[index] = crop
    return crops


def crop_view(
    crop: np.ndarray,
    angle: float,
    imgsz: int,
    scale: float = 1.0,
    shift: tuple[float, float] = (0.0, 0.0),
) -> np.ndarray:
    """crop을 가운데 기준으로 돌리고 키운 뒤 가운데 imgsz 영역을 자른다."""
    size = crop.shape[0]
    matrix = cv2.getRotationMatrix2D((size / 2, size / 2), angle, scale)
    matrix[:, 2] += np.asarray(shift) - (size - imgsz) / 2
    return cv2.warpAffine(crop, matrix, (imgsz, imgsz), borderMode=cv2.BORDER_REPLICATE)


def augment_crop(crop: np.ndarray, rng: np.random.Generator, imgsz: int) -> np.ndarray:
    """학습 crop을 임의 각도로 돌리고 bbox 오차만큼 크기·위치를 흔든 뒤 밝기·대비를 바꾼다."""
    view = crop_view(
        crop,
        angle=rng.uniform(0, 360),
        imgsz=imgsz,
        scale=rng.uniform(0.9, 1.1),
        shift=tuple(rng.uniform(-0.05, 0.05, 2) * imgsz),
    )
    return cv2.convertScaleAbs(view, alpha=rng.uniform(0.8, 1.2), beta=rng.uniform(-20, 20))


def crops_to_tensor(images: np.ndarray) -> torch.Tensor:
    """(N, H, W, 3) uint8 BGR을 (N, 3, H, W) 0~1 텐서로 바꿔 장치에 올린다."""
    return torch.from_numpy(images).permute(0, 3, 1, 2).float().div(255).to(DEVICE)


class ImprintOCR(torch.nn.Module):
    """알약 crop에서 각인 문자열을 읽는 CRNN(VGG형 CNN + BiLSTM). CTC로 학습한다."""

    def __init__(self, n_symbols: int, hidden: int = 256) -> None:
        super().__init__()

        def conv(in_channels: int, out_channels: int) -> list[torch.nn.Module]:
            return [
                torch.nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
                torch.nn.BatchNorm2d(out_channels),
                torch.nn.ReLU(inplace=True),
            ]

        # 128×128 입력 → 세로 8 × 가로 32. 가로를 덜 줄여 글자 순서를 남긴다.
        self.features = torch.nn.Sequential(
            *conv(3, 64),
            torch.nn.MaxPool2d(2),
            *conv(64, 128),
            torch.nn.MaxPool2d(2),
            *conv(128, 256),
            *conv(256, 256),
            torch.nn.MaxPool2d((2, 1)),
            *conv(256, 512),
            *conv(512, 512),
            torch.nn.MaxPool2d((2, 1)),
        )
        self.rnn = torch.nn.LSTM(
            512, hidden, num_layers=2, bidirectional=True, batch_first=True, dropout=0.1
        )
        self.classifier = torch.nn.Linear(2 * hidden, n_symbols + 1)  # 0번은 CTC blank

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """(B, 3, H, W) → 글자별 로그 확률 (B, T, 글자 수 + 1)."""
        features = self.features(images).mean(dim=2)  # 세로 평균 → (B, C, T)
        sequence, _ = self.rnn(features.transpose(1, 2))
        return self.classifier(sequence).log_softmax(-1)


class OcrCandidates:
    """모델 클래스마다 앞·뒷면 각인을 글자 번호로 바꾼 후보 표. class_ids 순서 = yolo_idx 순서."""

    def __init__(self, dictionary: pd.DataFrame, vocab: list[str], seq_len: int) -> None:
        self.class_ids = [int(value) for value in dictionary.index]
        self.texts = sorted({text for text in dictionary.to_numpy().ravel()})
        self.vocab = vocab
        self.seq_len = seq_len
        self.char_index = {char: index + 1 for index, char in enumerate(vocab)}
        text_index = {text: index for index, text in enumerate(self.texts)}
        # 클래스 × 면 → 후보 텍스트 번호
        self.class_texts = np.array(
            [[text_index[text] for text in row] for row in dictionary.to_numpy()]
        )
        too_long = [text for text in self.texts if ctc_length(text) > seq_len]
        if too_long:
            raise ValueError(f"OCR 출력 길이 {seq_len}보다 긴 각인이 있습니다: {too_long}")

    def encode(self, text: str) -> list[int]:
        """각인 문자열을 글자 번호 목록으로 바꾼다."""
        return [self.char_index[char] for char in text]


@torch.inference_mode()
def ocr_class_log_probs(
    model: ImprintOCR, crops: np.ndarray, candidates: OcrCandidates, ocr_cfg: dict
) -> np.ndarray:
    """crop마다 모델 클래스의 각인이 나올 CTC 로그 확률 (N, 클래스 수).

    crop을 rotations 방향으로 돌려 가장 잘 읽히는 방향을 쓰고, 앞·뒷면 중 더 맞는 면을 쓴다.
    """
    model.eval()
    rotations, imgsz = ocr_cfg["rotations"], ocr_cfg["imgsz"]
    angles = [360 * index / rotations for index in range(rotations)]
    targets = [torch.tensor(candidates.encode(text), dtype=torch.long) for text in candidates.texts]
    text_scores = []
    for start in range(0, len(crops), OCR_SCORE_BATCH):
        views = np.stack(
            [
                crop_view(crop, angle, imgsz)
                for crop in crops[start : start + OCR_SCORE_BATCH]
                for angle in angles
            ]
        )
        # CTC는 MPS에 없어 CPU에서 계산한다.
        log_probs = model(crops_to_tensor(views)).float().cpu().transpose(0, 1)  # (T, B, C)
        n_views = log_probs.shape[1]
        scores = torch.empty(n_views, len(targets))
        for index, target in enumerate(targets):
            scores[:, index] = -torch.nn.functional.ctc_loss(
                log_probs,
                target.repeat(n_views),
                input_lengths=torch.full((n_views,), log_probs.shape[0], dtype=torch.long),
                target_lengths=torch.full((n_views,), len(target), dtype=torch.long),
                reduction="none",
            )
        text_scores.append(scores.view(-1, len(angles), len(targets)).amax(dim=1))
    text_scores = torch.cat(text_scores).numpy()  # (N, 후보 텍스트 수)
    return text_scores[:, candidates.class_texts].max(axis=2)  # 면 중 큰 값 → (N, 클래스 수)


def ocr_probabilities(class_log_probs: np.ndarray, temperature: float) -> np.ndarray:
    """클래스 로그 확률을 temperature로 나눠 softmax한 OCR 확률."""
    logits = class_log_probs / temperature
    logits = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(logits)
    return probs / probs.sum(axis=1, keepdims=True)


def ocr_probs_for_boxes(
    model: ImprintOCR, boxes: pd.DataFrame, candidates: OcrCandidates, ocr_cfg: dict
) -> np.ndarray:
    """bbox 표(path, x, y, w, h)의 OCR 확률 (N, 클래스 수). crop은 OCR_CROP_CHUNK개씩 읽는다."""
    parts = []
    for start in tqdm(range(0, len(boxes), OCR_CROP_CHUNK), desc="OCR", leave=False):
        crops = load_crops(boxes.iloc[start : start + OCR_CROP_CHUNK], ocr_cfg["imgsz"])
        parts.append(ocr_class_log_probs(model, crops, candidates, ocr_cfg))
    if not parts:
        return np.zeros((0, len(candidates.class_ids)))
    return ocr_probabilities(np.concatenate(parts), ocr_cfg["temperature"])


def ocr_accuracy(
    model: ImprintOCR,
    crops: np.ndarray,
    categories: np.ndarray,
    candidates: OcrCandidates,
    ocr_cfg: dict,
) -> float:
    """정답 crop에서 OCR 확률이 가장 큰 클래스가 정답 클래스와 같은 비율."""
    scores = ocr_class_log_probs(model, crops, candidates, ocr_cfg)
    predicted = np.asarray(candidates.class_ids)[scores.argmax(axis=1)]
    return float((predicted == categories).mean())


def train_ocr(cfg: dict, fold: int, boxes: pd.DataFrame, candidates: OcrCandidates) -> Path:
    """학습 bbox crop으로 CRNN을 CTC로 학습하고 val 각인 정확도가 가장 높은 가중치를 반환한다."""
    ocr_cfg = cfg["ocr"]
    out_dir = ocr_dir(cfg, fold)
    config = {
        **{key: ocr_cfg[key] for key in OCR_TRAINING_KEYS},
        "fraction": cfg["fraction"],
        "seed": cfg["seed"],
        "fold": fold,
        "external_fingerprint": (
            get_external_bundle(cfg)["fingerprint"] if external_enabled(cfg) else None
        ),
    }
    complete_path, best_path = out_dir / "completed.json", out_dir / "best.pt"
    if complete_path.is_file():
        complete = json.loads(complete_path.read_text(encoding="utf-8"))
        if complete["config"] == json.loads(json.dumps(config)) and best_path.is_file():
            print(f"완료된 OCR 학습 사용: {best_path}")
            return best_path
        print("저장된 OCR 기록과 설정이 달라 OCR을 다시 학습합니다.")
    out_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(cfg["seed"])
    torch.manual_seed(cfg["seed"])
    imgsz, batch_size, epochs = ocr_cfg["imgsz"], ocr_cfg["batch"], ocr_cfg["epochs"]

    train = boxes[boxes["split"].eq("train") & boxes["text"].notna()]
    train = train[train["text"].map(ctc_length).le(candidates.seq_len)]
    val = boxes[boxes["split"].eq("val") & boxes["category_id"].isin(candidates.class_ids)]
    print(f"OCR crop 읽는 중: train {len(train)}개, val {len(val)}개")
    train_crops, val_crops = load_crops(train, imgsz), load_crops(val, imgsz)
    train_targets = [candidates.encode(text) for text in train["text"]]
    val_categories = val["category_id"].to_numpy()

    model = ImprintOCR(len(candidates.vocab)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=ocr_cfg["lr"], weight_decay=1e-4)
    steps_per_epoch = math.ceil(len(train) / batch_size)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=ocr_cfg["lr"], total_steps=epochs * steps_per_epoch, pct_start=0.1
    )
    history, best_accuracy, best_epoch = [], -1.0, 0
    for epoch in range(1, epochs + 1):
        model.train()
        started, losses = time.perf_counter(), []
        order = rng.permutation(len(train))
        for start in range(0, len(order), batch_size):
            batch = order[start : start + batch_size]
            images = crops_to_tensor(
                np.stack([augment_crop(train_crops[i], rng, imgsz) for i in batch])
            )
            targets = [train_targets[i] for i in batch]
            log_probs = model(images).float().cpu().transpose(0, 1)  # (T, B, C)
            loss = torch.nn.functional.ctc_loss(
                log_probs,
                torch.tensor([char for target in targets for char in target], dtype=torch.long),
                input_lengths=torch.full((len(batch),), log_probs.shape[0], dtype=torch.long),
                target_lengths=torch.tensor([len(target) for target in targets], dtype=torch.long),
                zero_infinity=True,
            )
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            scheduler.step()
            losses.append(float(loss))
        accuracy = ocr_accuracy(model, val_crops, val_categories, candidates, ocr_cfg)
        history.append(
            {
                "epoch": epoch,
                "train_ctc_loss": float(np.mean(losses)),
                "val_imprint_accuracy": accuracy,
                "lr": scheduler.get_last_lr()[0],
                "epoch_seconds": time.perf_counter() - started,
            }
        )
        print(json.dumps(history[-1], ensure_ascii=False))
        pd.DataFrame(history).to_csv(out_dir / "history.csv", index=False)
        if accuracy > best_accuracy:
            best_accuracy, best_epoch = accuracy, epoch
            torch.save(
                {"state_dict": model.state_dict(), "vocab": candidates.vocab, "epoch": epoch},
                best_path,
            )
        elif epoch - best_epoch >= ocr_cfg["patience"]:
            print(f"OCR early stopping: {ocr_cfg['patience']} epoch 개선 없음, best={best_epoch}")
            break
    complete_path.write_text(
        json.dumps(
            {
                "config": config,
                "best_epoch": best_epoch,
                "best_val_imprint_accuracy": best_accuracy,
                "train_crops": len(train),
                "val_crops": len(val),
                "vocab": candidates.vocab,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    del model
    gc.collect()
    empty_device_cache()
    return best_path


def load_ocr(path: Path, candidates: OcrCandidates) -> ImprintOCR:
    """저장한 OCR 가중치를 읽는다. 글자 목록이 지금 후보와 다르면 멈춘다."""
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    if checkpoint["vocab"] != candidates.vocab:
        raise ValueError("저장한 OCR의 글자 목록이 지금 각인 사전과 다릅니다.")
    model = ImprintOCR(len(candidates.vocab))
    model.load_state_dict(checkpoint["state_dict"])
    return model.to(DEVICE).eval()


def rerank_with_ocr(
    pred: pd.DataFrame, probs: np.ndarray, candidates: OcrCandidates, ocr_cfg: dict
) -> pd.DataFrame:
    """OCR이 다른 클래스를 강하게 가리킬 때만 클래스(category_id, yolo_idx)를 바꾼다.

    OCR 최고 클래스 확률 ≥ switch_prob이고 탐지기 클래스의 OCR 확률 ≤ keep_prob일 때다.
    """
    position = {code: index for index, code in enumerate(candidates.class_ids)}
    detector_column = pred["category_id"].map(position).astype(int).to_numpy()
    rows = np.arange(len(pred))
    best = probs.argmax(axis=1)
    switch = (
        (best != detector_column)
        & (probs[rows, best] >= ocr_cfg["switch_prob"])
        & (probs[rows, detector_column] <= ocr_cfg["keep_prob"])
    )
    result = pred.copy()
    result["category_id_detector"] = result["category_id"]
    result["category_id_ocr"] = np.asarray(candidates.class_ids)[best]
    result["ocr_prob_detector"] = probs[rows, detector_column]
    result["ocr_prob_best"] = probs[rows, best]
    result["ocr_switched"] = switch
    result.loc[switch, "category_id"] = result.loc[switch, "category_id_ocr"]
    result.loc[switch, "yolo_idx"] = best[switch]  # class_ids 순서 = yolo_idx 순서
    return result


def switch_outcomes(reranked: pd.DataFrame, gt: pd.DataFrame) -> dict[str, int]:
    """클래스를 바꾼 예측을 IoU 0.5 이상인 가장 가까운 정답과 비교해 바뀐 결과를 센다."""
    gt_by = dict(tuple(gt.groupby("file_name")))
    counts = {"wrong_to_right": 0, "right_to_wrong": 0, "wrong_to_wrong": 0, "no_match": 0}
    for row in reranked[reranked["ocr_switched"]].itertuples():
        g = gt_by.get(row.file_name)
        if g is None:
            counts["no_match"] += 1
            continue
        iou = box_iou(
            xywh_to_xyxy(np.array([[row.x, row.y, row.w, row.h]], dtype=float)),
            xywh_to_xyxy(g[BBOX_COLS].to_numpy(float)),
        )[0]
        if iou.max() < 0.5:
            counts["no_match"] += 1
            continue
        target = int(g["category_id"].iloc[iou.argmax()])
        if row.category_id == target:
            counts["wrong_to_right"] += 1
        elif row.category_id_detector == target:
            counts["right_to_wrong"] += 1
        else:
            counts["wrong_to_wrong"] += 1
    return counts


def plot_ocr_history(out_dir: Path) -> None:
    """OCR epoch별 학습 CTC loss와 val 각인 분류 정확도를 그린다."""
    history = pd.read_csv(out_dir / "history.csv")
    best = history.loc[history["val_imprint_accuracy"].idxmax()]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["epoch"], history["train_ctc_loss"], marker=".")
    axes[0].set(title="OCR train CTC loss", xlabel="epoch")
    axes[1].plot(history["epoch"], history["val_imprint_accuracy"], marker=".")
    axes[1].scatter(
        best["epoch"], best["val_imprint_accuracy"], marker="*", s=300, color="red", zorder=3
    )
    axes[1].set(title="OCR validation imprint accuracy", xlabel="epoch", ylim=(0, 1.05))
    for axis in axes:
        axis.axvline(best["epoch"], color="gray", linestyle=":")
        axis.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_dir / "ocr_history.png", dpi=150)
    plt.show()
    plt.close(fig)

In [ ]:
# 10절 본 학습(fold 0, 선형 감소)의 val 예측이 있으면 그 run, 없으면 9절 동작 확인 run에 붙인다.
OCR_FOLD = 0
OCR_RUN_CFG = {**CFG, "cos_lr": False}
if not (results_dir(OCR_RUN_CFG) / f"{run_stem(OCR_RUN_CFG, OCR_FOLD)}_val_pred.csv").is_file():
    OCR_RUN_CFG, OCR_FOLD = SMOKE_CFG, SMOKE_CFG["folds"][0]
OCR_CFG = OCR_RUN_CFG["ocr"]
print(f"OCR 대상 run: {OCR_RUN_CFG['exp']}/{run_stem(OCR_RUN_CFG, OCR_FOLD)}")

if not OCR_CFG["enabled"]:
    print('CFG["ocr"]["enabled"]=False: 각인 OCR을 건너뜁니다.')
else:
    model_classes = model_class_map(OCR_RUN_CFG)
    if model_classes["yolo_idx"].tolist() != list(range(len(model_classes))):
        raise ValueError("모델 클래스 표의 yolo_idx가 0부터 이어지지 않습니다.")
    OCR_BOXES = ocr_box_table(OCR_RUN_CFG, OCR_FOLD)
    OCR_DICTIONARY = imprint_dictionary(
        OCR_BOXES, model_classes["category_id"].astype(int).tolist()
    )
    OCR_VOCAB = sorted(
        set("".join(OCR_DICTIONARY.to_numpy().ravel())) | set("".join(OCR_BOXES["text"].dropna()))
    )
    OCR_CANDIDATES = OcrCandidates(OCR_DICTIONARY, OCR_VOCAB, OCR_CFG["imgsz"] // 4)
    text_known = OCR_BOXES["text"].notna()
    display(
        pd.DataFrame(
            {
                "bbox": OCR_BOXES.groupby(["source", "split"]).size(),
                "보이는 면 각인 있음": OCR_BOXES[text_known].groupby(["source", "split"]).size(),
                "빈 각인(글자 없는 면)": OCR_BOXES[text_known & OCR_BOXES["text"].eq("")]
                .groupby(["source", "split"])
                .size(),
            }
        )
        .fillna(0)
        .astype(int)
    )
    print(
        f"모델 클래스 {len(model_classes)}종, 서로 다른 각인 후보 {len(OCR_CANDIDATES.texts)}개, "
        f"글자 {len(OCR_VOCAB)}종: {''.join(OCR_VOCAB)}"
    )
    shared = OCR_DICTIONARY.reset_index().melt("category_id", value_name="text")
    shared = (
        shared[shared["text"].ne("")]
        .groupby("text")["category_id"]
        .agg(lambda ids: sorted(set(ids)))
    )
    shared = shared[shared.map(len) > 1]
    if len(shared):
        print(f"여러 클래스가 같은 각인을 쓰는 경우 {len(shared)}개 (OCR로 구별 불가):")
        display(shared.to_frame("category_id"))

    ocr_weights = train_ocr(OCR_RUN_CFG, OCR_FOLD, OCR_BOXES, OCR_CANDIDATES)
    OCR_OUT_DIR = ocr_dir(OCR_RUN_CFG, OCR_FOLD)
    OCR_DICTIONARY.to_csv(OCR_OUT_DIR / "imprint_dictionary.csv")
    plot_ocr_history(OCR_OUT_DIR)
    OCR_MODEL = load_ocr(ocr_weights, OCR_CANDIDATES)

    # 7절에서 저장한 val 예측(conf 0.001, 외부 전용 클래스 포함)에 OCR 재분류를 적용한다.
    ocr_stem = run_stem(OCR_RUN_CFG, OCR_FOLD)
    raw_val_pred = pd.read_csv(results_dir(OCR_RUN_CFG) / f"{ocr_stem}_val_pred.csv")
    ocr_val_files = fold_files(OCR_FOLD, "val")
    ocr_val_gt = LABELS[LABELS["file_name"].isin(ocr_val_files)]
    started = time.perf_counter()
    val_probs = ocr_probs_for_boxes(
        OCR_MODEL,
        raw_val_pred.assign(path=[TRAIN_IMG_DIR / name for name in raw_val_pred["file_name"]]),
        OCR_CANDIDATES,
        OCR_CFG,
    )
    ocr_seconds = time.perf_counter() - started
    VAL_RERANKED = rerank_with_ocr(raw_val_pred, val_probs, OCR_CANDIDATES, OCR_CFG)
    VAL_RERANKED.to_csv(
        OCR_OUT_DIR / f"val_reranked_conf{OCR_RUN_CFG['conf']:g}.csv", index=False, encoding="utf-8"
    )
    before = detection_metrics(competition_pred(raw_val_pred), ocr_val_gt, ocr_val_files)
    after = detection_metrics(competition_pred(VAL_RERANKED), ocr_val_gt, ocr_val_files)
    section7 = pd.read_csv(results_dir(OCR_RUN_CFG) / f"{ocr_stem}_summary.csv").iloc[0]
    gt_val = OCR_BOXES[OCR_BOXES["split"].eq("val")]
    OCR_RESULTS = {
        "val_imprint_accuracy_gt_crops": ocr_accuracy(
            OCR_MODEL,
            load_crops(gt_val, OCR_CFG["imgsz"]),
            gt_val["category_id"].to_numpy(),
            OCR_CANDIDATES,
            OCR_CFG,
        ),
        f"{MAIN_METRIC}_section7": float(section7[MAIN_METRIC]),
        f"{MAIN_METRIC}_detector_recomputed": before[MAIN_METRIC],
        f"{MAIN_METRIC}_detector_ocr": after[MAIN_METRIC],
        "delta": after[MAIN_METRIC] - before[MAIN_METRIC],
        "map50_detector": before["map50"],
        "map50_detector_ocr": after["map50"],
        "n_predictions": len(VAL_RERANKED),
        "n_switched": int(VAL_RERANKED["ocr_switched"].sum()),
        **switch_outcomes(VAL_RERANKED, ocr_val_gt),
        "ocr_ms_per_image": ocr_seconds * 1000 / max(len(ocr_val_files), 1),
        **{f"ocr_{key}": value for key, value in OCR_CFG.items()},
    }
    (OCR_OUT_DIR / "ocr_results.json").write_text(
        json.dumps(OCR_RESULTS, ensure_ascii=False, indent=2, default=float), encoding="utf-8"
    )
    display(pd.Series(OCR_RESULTS, name="OCR 재분류 (val, conf 0.001)").to_frame())
    print("7절 값과 다시 계산한 탐지기 값이 같아야 OCR 전후 비교를 믿을 수 있습니다.")
    switched = VAL_RERANKED[VAL_RERANKED["ocr_switched"]]
    if len(switched):
        display(
            switched[
                [
                    "file_name",
                    "conf",
                    "category_id_detector",
                    "category_id",
                    "ocr_prob_detector",
                    "ocr_prob_best",
                ]
            ]
        )

## 16. test 예측과 제출 csv

한 fold의 `best.pt`로 test 842장을 예측해 제출 형식
`annotation_id, image_id, category_id, bbox_x, bbox_y, bbox_w, bbox_h, score`로 저장한다.
`category_id`는 원본 JSON id, 좌표는 원본 픽셀 xywh, `image_id`는 파일 이름의 숫자다.
행은 image_id 순, 같은 사진 안에서는 score가 높은 순이고 conf 0.001 미만은 버린다.
외부 전용 클래스 예측은 `submit_extra = True`면 그 약 코드(AI Hub `K-` 코드 숫자)를 category_id로
넣고(B안), `False`면 버린다(A안). OCR 재분류 제출 csv는 16-2절에서 따로 만든다.

In [ ]:
SUBMISSION_COLS = [
    "annotation_id",
    "image_id",
    "category_id",
    "bbox_x",
    "bbox_y",
    "bbox_w",
    "bbox_h",
    "score",
]


def to_submission(pred: pd.DataFrame) -> pd.DataFrame:
    """예측 표를 대회 제출 형식으로 바꾼다(models/evaluate.py의 to_submission과 같음)."""
    sub = pred.rename(
        columns={"x": "bbox_x", "y": "bbox_y", "w": "bbox_w", "h": "bbox_h", "conf": "score"}
    )
    sub["image_id"] = sub["file_name"].map(lambda name: int(Path(name).stem))
    sub = sub.sort_values(["image_id", "score"], ascending=[True, False], ignore_index=True)
    sub["annotation_id"] = sub.index + 1
    return sub[SUBMISSION_COLS]


def make_submission(cfg: dict, fold: int, device: str = DEVICE) -> tuple[pd.DataFrame, Path]:
    """fold의 best.pt로 test를 예측해 제출 csv를 저장하고 (제출 표, 경로)를 반환한다."""
    stem = run_stem(cfg, fold)
    weights = Path(cfg["runs_dir"]) / cfg["exp"] / stem / "weights" / "best.pt"
    model = load_model(weights, cfg, device)
    test_files = sorted(
        (p.name for p in TEST_IMG_DIR.iterdir() if p.suffix.lower() in {".png", ".jpg"}),
        key=lambda name: int(Path(name).stem),
    )
    pred, _ = predict_images(model, TEST_IMG_DIR, test_files, device, cfg["eval_batch"], "test")
    pred = pred[pred["conf"] >= cfg["conf"]]
    if not cfg.get("external_data", {}).get("submit_extra", False):
        pred = competition_pred(pred)
    out_dir = results_dir(cfg)
    pred.to_csv(out_dir / f"{stem}_test_pred.csv", index=False, encoding="utf-8")
    sub = to_submission(pred)
    out_path = out_dir / f"{stem}_submission_conf{cfg['conf']:g}.csv"
    sub.to_csv(out_path, index=False, encoding="utf-8")
    return sub, out_path


RUN_SUBMISSION = False
if RUN_SUBMISSION:
    sub, sub_path = make_submission(REPORT_CFG, REPORT_FOLD)
    print(f"사진 {sub['image_id'].nunique()}장, 박스 {len(sub)}개: {sub_path}")
    display(sub.head())

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.ticker import MaxNLocator


def plot_local_training_history(csv_path: Path) -> None:
    """PC에 저장한 전체 에폭 기록으로 학습·검증 손실과 검증 지표를 그린다."""
    if not csv_path.is_file():
        raise FileNotFoundError(f"PC에 본 학습 results.csv를 복사해 주세요: {csv_path}")
    history = pd.read_csv(csv_path).sort_values("epoch")
    missing = sorted({"epoch", "map75_95"} - set(history.columns))
    if missing:
        raise ValueError(f"학습 기록에 필수 열이 없습니다: {missing}")
    if history.empty or history["map75_95"].notna().sum() == 0:
        raise ValueError("시각화할 에폭 또는 mAP75-95 기록이 없습니다.")

    best_row = history.loc[history["map75_95"].idxmax()]
    best_epoch = int(best_row["epoch"])
    marker = "o" if len(history) <= 15 else None
    loss_specs = [
        ("loss", "Total loss"),
        ("loss_classifier", "RoI classification loss"),
        ("loss_box_reg", "RoI box regression loss"),
        ("loss_objectness", "RPN objectness loss"),
        ("loss_rpn_box_reg", "RPN box regression loss"),
    ]
    # 최고 mAP75-95를 기록한 동일 에폭의 손실과 지표를 함께 출력한다.
    best_losses = pd.DataFrame(
        {
            label: {
                title: best_row.get(f"{split}/{key}", float("nan")) for key, title in loss_specs
            }
            for split, label in (("train", "Train"), ("val", "Validation"))
        }
    )
    best_metric_specs = [
        ("map75_95", "mAP75-95"),
        ("map50", "mAP50"),
        ("map75", "mAP75"),
        ("map50_95", "mAP50-95"),
        ("ul_precision", "Precision (macro, max-F1)"),
        ("ul_recall", "Recall (macro, max-F1)"),
    ]
    best_metrics = pd.DataFrame(
        {
            "Value": {
                title: best_row[key] for key, title in best_metric_specs if key in history.columns
            }
        }
    )
    print(f"{csv_path.parent.name} | 최고 mAP75-95 에폭: {best_epoch}")
    print("최고 에폭의 학습·검증 손실")
    display(best_losses.round(6))
    print("동일 에폭의 검증 평가지표")
    display(best_metrics.round(6))

    panels = [
        (title, [(f"train/{key}", "Train"), (f"val/{key}", "Validation")], False)
        for key, title in loss_specs
    ]
    panels += [
        ("Validation mAP75-95", [("map75_95", "mAP75-95")], True),
        (
            "Validation other mAP",
            [("map50", "mAP50"), ("map75", "mAP75"), ("map50_95", "mAP50-95")],
            True,
        ),
        ("Validation precision (macro, max-F1)", [("ul_precision", "Precision")], True),
        ("Validation recall (macro, max-F1)", [("ul_recall", "Recall")], True),
    ]

    fig, axes = plt.subplots(3, 3, figsize=(18, 13))
    for axis, (title, columns, is_metric) in zip(axes.flat, panels, strict=True):
        for column, label in columns:
            if column in history.columns:
                axis.plot(
                    history["epoch"],
                    history[column],
                    label=label,
                    marker=marker,
                    linewidth=1.8,
                )
        axis.axvline(best_epoch, color="gray", linestyle="--", label=f"Best epoch {best_epoch}")
        axis.set(
            title=title,
            xlabel="Epoch",
            ylabel="Score" if is_metric else "Loss",
            xlim=(history["epoch"].min() - 0.5, history["epoch"].max() + 0.5),
        )
        if is_metric:
            axis.set_ylim(-0.02, 1.02)
        axis.xaxis.set_major_locator(MaxNLocator(integer=True))
        axis.grid(alpha=0.3)
        axis.legend(fontsize=8)

    fig.suptitle(
        f"{csv_path.parent.name}\nBest epoch: {best_epoch} | mAP75-95: {best_row['map75_95']:.4f}"
    )
    fig.tight_layout()
    plt.show()
    plt.close(fig)


# 프로젝트 밖에서 실행한다면 저장소 경로를 직접 지정할 수 있다.
VISUALIZATION_PROJECT_ROOT: Path | None = None
if VISUALIZATION_PROJECT_ROOT is not None:
    LOCAL_ROOT_CANDIDATES = [Path(VISUALIZATION_PROJECT_ROOT).expanduser()]
else:
    LOCAL_ROOT_CANDIDATES = [
        Path.cwd().resolve(),
        *Path.cwd().resolve().parents,
        Path.cwd() / "pill-bbox",
        Path("/content/pill-bbox"),
    ]
LOCAL_PROJECT_ROOT = next(
    (candidate for candidate in LOCAL_ROOT_CANDIDATES if (candidate / "pyproject.toml").is_file()),
    None,
)
if LOCAL_PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"프로젝트 경로를 찾지 못했습니다. 현재 작업 폴더: {Path.cwd()}\n"
        "PC에서는 로컬 Python 커널을 선택하고, 필요하면 "
        "VISUALIZATION_PROJECT_ROOT에 저장소 경로를 지정하세요."
    )
print(f"시각화 기준 경로: {LOCAL_PROJECT_ROOT}")

# Colab의 본 학습 기록을 PC의 동일한 runs 하위 폴더로 복사한다.
LOCAL_HISTORY_ROOT = LOCAL_PROJECT_ROOT / "runs" / CFG["exp"]
LOCAL_HISTORY_RUN = run_stem({**CFG, "cos_lr": False}, 0).rsplit("_fold", 1)[0]
LOCAL_HISTORY_PATHS = sorted(LOCAL_HISTORY_ROOT.glob(f"{LOCAL_HISTORY_RUN}_fold*/results.csv"))
if not LOCAL_HISTORY_PATHS:
    raise FileNotFoundError(
        f"본 학습 기록({LOCAL_HISTORY_RUN})이 없습니다. Colab의 results.csv를 "
        f"PC의 동일한 run 폴더에 복사하세요: {LOCAL_HISTORY_ROOT}"
    )
for history_path in LOCAL_HISTORY_PATHS:
    plot_local_training_history(history_path)

In [ ]:
import json
from pathlib import Path

import numpy as np

# 현재 본 학습 설정: 선형 감소, fold 0
submission_cfg = {**CFG, "cos_lr": False}
submission_fold = 0

# 파일 이름은 여기서 변경하면 된다.
output_path = PROJECT_ROOT / "submission_faster_rcnn_aihub_ocr.csv"

weights = (
    Path(submission_cfg["runs_dir"])
    / submission_cfg["exp"]
    / run_stem(submission_cfg, submission_fold)
    / "weights"
    / "best.pt"
)

if not weights.is_file():
    raise FileNotFoundError(f"학습된 가중치가 없습니다: {weights}")

if output_path.exists():
    raise FileExistsError(f"같은 파일이 있습니다. 출력 이름을 변경하세요: {output_path}")

# 기본 데이터와 외부 데이터를 함께 학습한 실험인지 확인한다.
config_path = weights.parent.parent / "config.json"
saved_config = json.loads(config_path.read_text(encoding="utf-8"))

if saved_config.get("n_external_train_images", 0) <= 0:
    raise ValueError("외부 데이터를 함께 학습한 모델인지 확인하세요.")

test_files = sorted(
    (
        path.name
        for path in TEST_IMG_DIR.iterdir()
        if path.suffix.lower() in {".png", ".jpg", ".jpeg"}
    ),
    key=lambda name: int(Path(name).stem),
)

if not test_files:
    raise FileNotFoundError(f"test 이미지가 없습니다: {TEST_IMG_DIR}")

image_ids = [int(Path(name).stem) for name in test_files]
if len(image_ids) != len(set(image_ids)):
    raise ValueError("서로 다른 test 파일의 image_id가 중복됩니다.")

# 저장된 best.pt를 불러와 test 이미지만 예측한다.
print("사용 가중치:", weights)
print(
    f"학습 데이터: 기본 {saved_config['n_original_train_images']:,}장"
    f" + 외부 {saved_config['n_external_train_images']:,}장"
)

submission_model = load_model(weights, submission_cfg, DEVICE)
try:
    predictions, _ = predict_images(
        submission_model,
        TEST_IMG_DIR,
        test_files,
        DEVICE,
        submission_cfg["eval_batch"],
        "Kaggle test prediction",
    )
finally:
    del submission_model

# 기존 실험의 신뢰도 기준을 그대로 적용한다.
predictions = predictions.loc[predictions["conf"] >= submission_cfg["conf"]].copy()
# 외부 전용 클래스 예측: submit_extra면 약 코드를 category_id로 넣고(B안), 아니면 버린다(A안).
if not submission_cfg["external_data"]["submit_extra"]:
    predictions = competition_pred(predictions).copy()
print(
    f"외부 전용 클래스 예측 {int((predictions['yolo_idx'] >= len(CLASS_MAP)).sum()):,}개 포함 "
    f"(submit_extra={submission_cfg['external_data']['submit_extra']})"
)

if predictions.empty:
    raise ValueError("예측 객체가 없습니다. 가중치와 test 데이터를 확인하세요.")

# predict_images의 category_id는 대회 약제 ID이고, 외부 전용 클래스는 AI Hub 약 코드다.
submission = predictions.rename(
    columns={
        "x": "bbox_x",
        "y": "bbox_y",
        "w": "bbox_w",
        "h": "bbox_h",
        "conf": "score",
    }
)
submission["image_id"] = submission["file_name"].map(lambda name: int(Path(name).stem))
submission["category_id"] = submission["category_id"].astype("int64")
submission = submission.sort_values(
    ["image_id", "score"],
    ascending=[True, False],
    ignore_index=True,
)

# 검출 객체 하나당 한 행, annotation_id는 전체 행에서 고유하게 지정한다.
submission["annotation_id"] = np.arange(1, len(submission) + 1)

submission_columns = [
    "annotation_id",
    "image_id",
    "category_id",
    "bbox_x",
    "bbox_y",
    "bbox_w",
    "bbox_h",
    "score",
]
submission = submission[submission_columns]

# bbox는 원본 이미지의 픽셀 단위 [x, y, width, height]이다.
numeric = submission.to_numpy(dtype=float)
if not np.isfinite(numeric).all():
    raise ValueError("제출 데이터에 결측 또는 무한대 값이 있습니다.")
if (
    submission[["bbox_x", "bbox_y"]].lt(0).any().any()
    or submission[["bbox_w", "bbox_h"]].le(0).any().any()
    or not submission["score"].between(0, 1).all()
):
    raise ValueError("bbox 좌표·크기 또는 score 범위가 잘못되었습니다.")

# 인덱스 열 없이 제출 CSV 하나만 저장한다.
submission.to_csv(output_path, index=False, encoding="utf-8")

print(f"제출 CSV: {output_path}")
print(
    f"test {len(test_files):,}장 / "
    f"검출된 이미지 {submission['image_id'].nunique():,}장 / "
    f"검출 객체 {len(submission):,}개"
)
display(submission.head())

### 16-2. OCR 재분류 제출 csv

위 제출 csv의 박스마다 15절 OCR 재분류를 적용한 csv를 따로 만든다. 박스·점수는 그대로이고 클래스만 바뀐다.
`submit_extra = False`면 외부 전용 클래스로 바뀐 박스는 지운다. 파일 이름은 위 제출 파일 이름 뒤에 `_ocr`을 붙인다.
어느 쪽을 낼지는 15절 val 비교를 보고 정한다. OCR은 위 제출과 같은 run·fold로 학습한 것이어야 한다.

In [ ]:
if not OCR_CFG["enabled"]:
    print('CFG["ocr"]["enabled"]=False: OCR 제출 csv를 만들지 않습니다.')
else:
    if (OCR_RUN_CFG["exp"], run_stem(OCR_RUN_CFG, OCR_FOLD)) != (
        submission_cfg["exp"],
        run_stem(submission_cfg, submission_fold),
    ):
        raise ValueError(
            "15절 OCR과 제출의 run·fold가 다릅니다. 같은 run으로 15절을 다시 실행하세요."
        )
    ocr_output_path = output_path.with_name(f"{output_path.stem}_ocr{output_path.suffix}")
    if ocr_output_path.exists():
        raise FileExistsError(f"같은 파일이 있습니다. 출력 이름을 변경하세요: {ocr_output_path}")
    test_probs = ocr_probs_for_boxes(
        OCR_MODEL,
        predictions.assign(path=[TEST_IMG_DIR / name for name in predictions["file_name"]]),
        OCR_CANDIDATES,
        OCR_CFG,
    )
    test_reranked = rerank_with_ocr(predictions, test_probs, OCR_CANDIDATES, OCR_CFG)
    test_reranked.to_csv(OCR_OUT_DIR / "test_reranked.csv", index=False, encoding="utf-8")
    if not submission_cfg["external_data"]["submit_extra"]:
        test_reranked = competition_pred(test_reranked)
    ocr_submission = to_submission(test_reranked[PRED_COLS])
    ocr_submission.to_csv(ocr_output_path, index=False, encoding="utf-8")
    print(
        f"OCR로 클래스를 바꾼 test 박스 {int(test_reranked['ocr_switched'].sum()):,}"
        f"/{len(predictions):,}개"
    )
    print(f"OCR 제출 CSV: {ocr_output_path}")
    display(ocr_submission.head())